# LC25000 + LungHist700 — Complete Reproducible Pipeline

**Everything from A to Z in one notebook.** Run the cells in order from top to bottom.

| Section | What it does |
|---|---|
| 0 | Configuration — **edit your paths here only** |
| 1 | Setup, imports, seeds, device |
| 2 | Dataset discovery and verification |
| 3 | **Leakage analysis and group-aware splitting** (reviewer issue A) |
| 4 | Transforms and DataLoaders (both protocols) |
| 5 | All 12 model architectures |
| 6 | Training and evaluation utilities |
| 7 | Train on the original image-level split |
| 8 | Train on the leakage-resistant split |
| 9 | Dual-protocol comparison |
| 10 | Full metrics, confusion matrices, ROC, per-class |
| 11 | TP / FP / TN / FN analysis |
| 12 | Stain-robustness evaluation |
| 13 | Efficiency and Pareto analysis |
| 14 | LungHist700 — cross-dataset and fine-tuned |
| 15 | Statistical analysis (McNemar, confidence intervals) |
| 16 | Export everything for the manuscript |

**Resume-aware.** Every training step checks for existing weights first, so
re-running is safe and already-trained models are skipped instantly.

**Windows note.** `NUM_WORKERS` is fixed at 0 throughout — Windows uses spawn-based
multiprocessing which crashes PyTorch DataLoader workers.

---
# 0. Configuration

**This is the only cell you need to edit.**

In [ ]:
from pathlib import Path

# ─── PATHS ────────────────────────────────────────────────────────────
LC25000_ROOT = Path(r"E:\LC25000")          # contains the class sub-folders
LUNGHIST_DIR = Path(r"E:\LungHist700")      # contains aca / nor / ssc
OUT_DIR      = Path(r"E:\LC25000_results")  # all outputs land here

# ─── EXPERIMENT SETTINGS ──────────────────────────────────────────────
SEED          = 42
IMG_SIZE      = 224
BATCH_SIZE    = 32
EPOCHS        = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-5
LR_STEP_SIZE  = 7
LR_GAMMA      = 0.1
NUM_WORKERS   = 0        # MUST be 0 on Windows

CLASS_NAMES   = ['colon_aca', 'colon_n', 'lung_aca', 'lung_n', 'lung_scc']
NUM_CLASSES   = len(CLASS_NAMES)

# ─── LEAKAGE-SPLIT SETTINGS ───────────────────────────────────────────
N_ORIGINALS_PER_CLASS = 250    # LC25000 ground truth (Borkowski et al. 2019)
MERGE_THRESHOLD       = 0.98   # calibrated by the sweep in Section 3
REPORT_THRESHOLD      = 0.98   # leakage level we claim to eliminate
KNN_K                 = 40

# ─── STAIN PERTURBATION ───────────────────────────────────────────────
JITTER_BCS = 0.35    # brightness / contrast / saturation
JITTER_HUE = 0.08

# ─── RETRAIN FLAGS (set True to force retraining) ─────────────────────
FORCE_RETRAIN_ORIG = False
FORCE_RETRAIN_LR   = False
FORCE_RETRAIN_LH   = False

# ─── derived output dirs ──────────────────────────────────────────────
LR_OUT_DIR = OUT_DIR / "leakage_resistant"
LH_OUT_DIR = OUT_DIR / "lunghist700"
for d in [OUT_DIR, LR_OUT_DIR, LH_OUT_DIR]:
    (d / "models").mkdir(parents=True, exist_ok=True)
    (d / "metrics").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(LR_OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  LC25000 : {LC25000_ROOT}")
print(f"  LungHist: {LUNGHIST_DIR}")
print(f"  Output  : {OUT_DIR}")

---
# 1. Setup — imports, seeds, device

In [ ]:
import torch.nn.functional as F
import gc
print("F and gc restored")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import os, json, random, time, re, warnings
from collections import Counter, defaultdict
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, matthews_corrcoef,
                             confusion_matrix, roc_curve, auc)
from sklearn.preprocessing import label_binarize
from sklearn.cluster import AgglomerativeClustering
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# reproducibility
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
print(f"Device         : {device}")
print(f"Seed           : {SEED}")

In [ ]:
import timm
print(f"timm : {timm.__version__}")
print("All packages OK")

---
# 2. Dataset discovery and verification

Pools **every** LC25000 image, regardless of how the folders are arranged.
This matters: the original train/test folders were themselves produced by an
image-level split, so we re-split from scratch at the group level in Section 3.

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

def collect_lc25000(root):
    # Walk the whole tree and keep any file sitting inside a folder
    # whose name matches one of CLASS_NAMES.
    records = []
    for p in root.rglob("*"):
        if p.suffix.lower() in IMG_EXTS and p.parent.name in CLASS_NAMES:
            records.append({"path": str(p), "class": p.parent.name})
    return pd.DataFrame(records)

lc_df = collect_lc25000(LC25000_ROOT)
assert len(lc_df) > 0, f"No images found under {LC25000_ROOT}"

print(f"Total images: {len(lc_df)}")
print(lc_df['class'].value_counts().sort_index().to_string())

if len(lc_df) != 25000:
    print(f"\nNOTE: expected 25000 images, found {len(lc_df)}.")
    print("Check that LC25000_ROOT points at the dataset root.")

In [ ]:
# Preview one image per class
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(16, 3.4))
for ax, cls in zip(axes, CLASS_NAMES):
    p = lc_df[lc_df['class'] == cls].iloc[0]['path']
    ax.imshow(Image.open(p).convert("RGB"))
    ax.set_title(cls, fontsize=11)
    ax.axis('off')
plt.suptitle("LC25000 — one sample per class", y=1.03)
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "sample_images.png", dpi=600, bbox_inches='tight')
plt.show()

---
# 3. Leakage analysis and group-aware splitting

**Reviewer issue A (R1-1/2/3, R4-1, R5-1).**

LC25000's 25,000 images derive from only **1,250 originals** (250 per class) via
rotation/flip augmentation. Filenames do not encode source identity, so an
image-level random split places augmented siblings of the same biopsy in both
train and test.

Pipeline: embed → cluster into the known 250 groups per class → conservative
merge of near-duplicates → `StratifiedGroupKFold` → verify → quantify.

### 3.1 Embed all images with a frozen backbone

Deep embeddings, not pixel hashes: the augmentation includes **rotation**, which
destroys pHash/dHash similarity but leaves semantic content intact.

In [ ]:
class PathDataset(Dataset):
    def __init__(self, paths, tf):
        self.paths, self.tf = paths, tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), i


EMB_PATH = OUT_DIR / "lc25000_embeddings.npy"

@torch.no_grad()
def embed_all(paths, model_name="convnext_tiny"):
    tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    m = timm.create_model(model_name, pretrained=True,
                          num_classes=0, global_pool="avg").to(device).eval()
    loader = DataLoader(PathDataset(paths, tf), batch_size=64,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)
    feats = np.zeros((len(paths), m.num_features), dtype=np.float32)
    for x, idx in tqdm(loader, desc="embedding"):
        feats[idx.numpy()] = m(x.to(device)).cpu().numpy()
    feats /= (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-8)
    return feats


if EMB_PATH.exists():
    feats = np.load(EMB_PATH)
    feats /= (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-8)
    print(f"Loaded cached embeddings: {feats.shape}")
else:
    feats = embed_all(lc_df["path"].tolist())
    np.save(EMB_PATH, feats)
    print(f"Saved embeddings: {feats.shape}")

### 3.2 Recover source groups by clustering

We do not guess the number of groups — LC25000's source paper documents 250
originals per class, so `n_clusters` is a known prior.

In [ ]:
def cluster_groups(df, feats, k=N_ORIGINALS_PER_CLASS):
    gid = np.full(len(df), -1, dtype=int)
    nxt, rep = 0, []
    for cls in CLASS_NAMES:
        idx = np.where((df["class"] == cls).values)[0]
        if len(idx) == 0:
            continue
        kk = min(k, len(idx))
        lab = AgglomerativeClustering(n_clusters=kk, linkage="ward").fit_predict(feats[idx])
        gid[idx] = lab + nxt
        nxt += kk
        sizes = list(Counter(lab).values())
        rep.append({"class": cls, "n_images": len(idx), "n_groups": kk,
                    "median_size": float(np.median(sizes)),
                    "min_size": int(min(sizes)), "max_size": int(max(sizes))})
    df = df.copy(); df["group_id"] = gid
    return df, pd.DataFrame(rep)


lc_df, cluster_report = cluster_groups(lc_df, feats)
print("Cluster recovery (median should be near 20 = 25000/1250):")
print(cluster_report.to_string(index=False))
cluster_report.to_csv(OUT_DIR / "cluster_quality_report.csv", index=False)

### 3.3 Conservative merge and threshold sweep

Fixed-k clustering can *fragment* one source across two clusters, and fragments
could land in different folds. The merge pass joins clusters containing mutually
near-duplicate images. Merging is the safe direction: over-grouping costs a
little diversity but cannot create leakage.

The sweep calibrates the threshold rather than assuming one — this table belongs
in the supplementary material.

In [ ]:
class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[rb] = ra


@torch.no_grad()
def merge_at(df, feats, threshold, k=KNN_K):
    uf = UnionFind(len(df))
    first = {}
    for i, g in enumerate(df["group_id"].values):
        if g not in first: first[g] = i
        else: uf.union(first[g], i)
    for cls in df["class"].unique():
        idx = np.where((df["class"] == cls).values)[0]
        X = torch.from_numpy(feats[idx]).float().to(device)
        for s in range(0, len(idx), 512):
            ch = X[s:s+512]
            sims = ch @ X.T
            for r in range(ch.shape[0]):
                sims[r, s + r] = -1.0
            tv, ti = torch.topk(sims, k=min(k, sims.shape[1]), dim=1)
            hit = (tv > threshold).cpu().numpy(); ti = ti.cpu().numpy()
            for r in range(ch.shape[0]):
                for c in np.where(hit[r])[0]:
                    uf.union(idx[s + r], idx[ti[r, c]])
    merged = np.array([uf.find(i) for i in range(len(df))])
    remap = {g: i for i, g in enumerate(sorted(set(merged)))}
    return np.array([remap[g] for g in merged])


def balanced_split(df, gcol, seed=SEED):
    y, g, idx = df["class"].values, df[gcol].values, np.arange(len(df))
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    tv_rel, te_rel = next(s1.split(idx, y, g))
    tv, te = idx[tv_rel], idx[te_rel]
    s2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(tv, y[tv], g[tv]))
    return tv[tr_rel], tv[va_rel], te


@torch.no_grad()
def leak_pct(feats, tr, te, thr, chunk=256):
    Xtr = torch.from_numpy(feats[tr]).float().to(device)
    hits = 0
    for s in range(0, len(te), chunk):
        b = torch.from_numpy(feats[te[s:s+chunk]]).float().to(device)
        hits += int(((b @ Xtr.T).max(dim=1).values > thr).sum().item())
    return 100.0 * hits / len(te)

In [ ]:
SWEEP = [0.970, 0.975, 0.980, 0.985, 0.990]
rows, cache = [], {}
for t in SWEEP:
    gm = merge_at(lc_df, feats, t); cache[t] = gm
    d2 = lc_df.copy(); d2["gm"] = gm
    tr, va, te = balanced_split(d2, "gm")
    sizes = list(Counter(gm).values())
    fold = np.empty(len(d2), dtype=object)
    fold[tr] = "train"; fold[va] = "val"; fold[te] = "test"
    tab = pd.crosstab(d2["class"], fold)
    frac = (tab["train"] / tab.sum(axis=1))
    rows.append({"threshold": t, "n_groups": len(set(gm)), "max_cluster": max(sizes),
                 "median_cluster": float(np.median(sizes)),
                 "min_train_frac": round(float(frac.min()), 3),
                 "max_train_frac": round(float(frac.max()), 3),
                 "leak_pct": round(leak_pct(feats, tr, te, REPORT_THRESHOLD), 3)})
    print(f"  t={t:.3f}  groups={rows[-1]['n_groups']:5d}  "
          f"max={rows[-1]['max_cluster']:5d}  leak={rows[-1]['leak_pct']:.2f}%")

sweep_df = pd.DataFrame(rows)
sweep_df.to_csv(OUT_DIR / "merge_threshold_sweep.csv", index=False)
print()
print(sweep_df.to_string(index=False))

In [ ]:
# Choose: zero residual leakage first, then the smallest maximum cluster
ok = sweep_df[sweep_df["leak_pct"] <= 0.0]
if len(ok) == 0:
    ok = sweep_df.nsmallest(1, "leak_pct")
best = ok.nsmallest(1, "max_cluster").iloc[0]
CHOSEN_T = float(best["threshold"])
print(f"Selected merge threshold: {CHOSEN_T}  "
      f"(max_cluster={int(best['max_cluster'])}, leak={best['leak_pct']}%)")

lc_df["group_final"] = cache[CHOSEN_T]
tr_i, va_i, te_i = balanced_split(lc_df, "group_final")
fold = np.empty(len(lc_df), dtype=object)
fold[tr_i] = "train"; fold[va_i] = "val"; fold[te_i] = "test"
lc_df["fold"] = fold

assert (lc_df.groupby("group_final")["fold"].nunique() == 1).all(), "LEAKAGE!"
print("Verified: zero groups span folds")
print()
print(pd.crosstab(lc_df["class"], lc_df["fold"]).to_string())

lc_df.to_csv(OUT_DIR / "lc25000_final_split.csv", index=False)
print(f"\nSaved final split -> {OUT_DIR/'lc25000_final_split.csv'}")

### 3.4 Quantify leakage — the number for the manuscript

For every test image, the maximum cosine similarity to **any** training image,
under both protocols.

In [ ]:
@torch.no_grad()
def max_sim_to_train(feats, tr, te, chunk=256):
    Xtr = torch.from_numpy(feats[tr]).float().to(device)
    out = np.zeros(len(te), dtype=np.float32)
    for s in range(0, len(te), chunk):
        b = torch.from_numpy(feats[te[s:s+chunk]]).float().to(device)
        out[s:s+chunk] = (b @ Xtr.T).max(dim=1).values.cpu().numpy()
    return out

# original image-level random split, for comparison
_idx = np.arange(len(lc_df))
_tv, _te_old = train_test_split(_idx, test_size=0.20, random_state=SEED,
                                stratify=lc_df["class"].values)
_tr_old, _ = train_test_split(_tv, test_size=0.20, random_state=SEED,
                              stratify=lc_df["class"].values[_tv])

sims_old = max_sim_to_train(feats, _tr_old, _te_old)
sims_new = max_sim_to_train(feats, tr_i, te_i)

leak_rows = []
for t in [0.90, 0.95, 0.98, 0.99]:
    leak_rows.append({"threshold": t,
                      "original_split_pct": round(100.0*float((sims_old > t).mean()), 2),
                      "group_split_pct":    round(100.0*float((sims_new > t).mean()), 2)})
leak_df = pd.DataFrame(leak_rows)
leak_df.to_csv(OUT_DIR / "leakage_quantification.csv", index=False)
print(leak_df.to_string(index=False))
print(f"\nMean max-similarity — original {sims_old.mean():.4f} | group-aware {sims_new.mean():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.linspace(0.5, 1.0, 80)
ax.hist(sims_old, bins=bins, alpha=0.65, density=True,
        label="Original image-level split", color="#C0392B")
ax.hist(sims_new, bins=bins, alpha=0.65, density=True,
        label="Group-aware split", color="#1C7293")
ax.axvline(0.98, color="k", ls="--", lw=1, label="Near-duplicate threshold (0.98)")
ax.set_xlabel("Max cosine similarity of a test image to any training image")
ax.set_ylabel("Density")
ax.set_title("Train-test near-duplicate leakage in LC25000")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "leakage_comparison.png", dpi=600, bbox_inches='tight')
plt.show()

---
# 4. Transforms and DataLoaders

Two protocols are built side by side so every result can be reported under both.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# stain-perturbed evaluation transform (Section 12)
stain_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=JITTER_BCS, contrast=JITTER_BCS,
                           saturation=JITTER_BCS, hue=JITTER_HUE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
print("Transforms ready")

In [ ]:
class SplitDataset(Dataset):
    # Reads rows of the split dataframe for one fold.
    def __init__(self, df, fold, class_names, transform=None):
        self.rows = df[df["fold"] == fold].reset_index(drop=True)
        self.c2i = {c: i for i, c in enumerate(class_names)}
        self.transform = transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows.iloc[i]
        img = Image.open(r["path"]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, self.c2i[r["class"]]


def make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=False)

# ---- PROTOCOL B: leakage-resistant (group-aware) ----
lr_train_loader = make_loader(SplitDataset(lc_df, "train", CLASS_NAMES, train_transform), True)
lr_val_loader   = make_loader(SplitDataset(lc_df, "val",   CLASS_NAMES, eval_transform),  False)
lr_test_loader  = make_loader(SplitDataset(lc_df, "test",  CLASS_NAMES, eval_transform),  False)
lr_stain_loader = make_loader(SplitDataset(lc_df, "test",  CLASS_NAMES, stain_transform), False)

# ---- PROTOCOL A: ORIGINAL folder-based split (matches the manuscript) ----
# LC25000 ships with "Train and Validation Set" and "Test Set" folders.
# Recover that split from the file paths so cached weights are evaluated on
# the exact test set they were held out from.
def _orig_fold_from_path(p):
    s = str(p).lower()
    if "test" in s and "set" in s:
        return "test"
    return "trainval"

_src = lc_df["path"].map(_orig_fold_from_path)

if (_src == "test").sum() > 0:
    print(f"Using LC25000's original folder split "
          f"({(_src=='trainval').sum()} train+val / {(_src=='test').sum()} test)")
    _tv_idx = np.where((_src == "trainval").values)[0]
    _te_old = np.where((_src == "test").values)[0]
    _tr_old, _va_old = train_test_split(
        _tv_idx, test_size=0.20, random_state=SEED,
        stratify=lc_df["class"].values[_tv_idx])
else:
    print("Original folder split not detected — falling back to a random split")
    _idx = np.arange(len(lc_df))
    _tv_idx, _te_old = train_test_split(_idx, test_size=0.20, random_state=SEED,
                                        stratify=lc_df["class"].values)
    _tr_old, _va_old = train_test_split(_tv_idx, test_size=0.20, random_state=SEED,
                                        stratify=lc_df["class"].values[_tv_idx])

orig_fold = np.empty(len(lc_df), dtype=object)
orig_fold[_tr_old] = "train"
orig_fold[_va_old] = "val"
orig_fold[_te_old] = "test"
orig_df = lc_df.copy(); orig_df["fold"] = orig_fold

or_train_loader = make_loader(SplitDataset(orig_df, "train", CLASS_NAMES, train_transform), True)
or_val_loader   = make_loader(SplitDataset(orig_df, "val",   CLASS_NAMES, eval_transform),  False)
or_test_loader  = make_loader(SplitDataset(orig_df, "test",  CLASS_NAMES, eval_transform),  False)
or_stain_loader = make_loader(SplitDataset(orig_df, "test",  CLASS_NAMES, stain_transform), False)

print("Protocol A (original folders): "
      f"train {len(or_train_loader.dataset)} | val {len(or_val_loader.dataset)} | test {len(or_test_loader.dataset)}")
print("Protocol B (leakage-resistant): "
      f"train {len(lr_train_loader.dataset)} | val {len(lr_val_loader.dataset)} | test {len(lr_test_loader.dataset)}")

---
# 5. Model architectures — all 12

Four families: classic CNN, efficient CNN, modern/transformer, hybrid.
`FSPAN_Y` is an existing hybrid; `DPCT_Net` and `MSCA_Net` are the proposed ones.

In [ ]:
class YBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dilation=2):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=dilation, dilation=dilation)
        self.bn2   = nn.BatchNorm2d(out_ch)
    def forward(self, x):
        a = F.relu(self.bn1(self.conv1(x)), inplace=True)
        b = F.relu(self.bn2(self.conv2(a)), inplace=True)
        return F.relu(a + b, inplace=True)


class FSPAN_Y(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = timm.create_model('convnext_tiny', pretrained=True,
                                          features_only=True, out_indices=(3,))
        for p in self.backbone.parameters(): p.requires_grad = False
        ch = self.backbone.feature_info.channels()[-1]
        self.bottleneck = nn.Sequential(nn.Conv2d(ch, 512, 1), nn.BatchNorm2d(512), nn.ReLU())
        self.attention = nn.Sequential(
            nn.Conv2d(512, 64, 1), nn.ReLU(), nn.Conv2d(64, 16, 1), nn.ReLU(),
            nn.Conv2d(16, 1, 1), nn.Sigmoid())
        self.y1 = YBlock(512, 256); self.y2 = YBlock(256, 128); self.y3 = YBlock(128, 64)
        self.gap = nn.AdaptiveAvgPool2d(1); self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(64, num_classes)
    def forward(self, x):
        f = self.backbone(x)[0]
        f = self.bottleneck(f)
        f = f * self.attention(f)
        f = self.y3(self.y2(self.y1(f)))
        return self.fc(self.dropout(self.gap(f).flatten(1)))

In [ ]:
class DPCT_Net(nn.Module):
    # Dual-Path CNN-Transformer with cross-attention and adaptive gating
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.cnn = timm.create_model('convnext_tiny', pretrained=True,
                                     num_classes=0, global_pool='avg')
        self.tfm = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True,
                                     num_classes=0, global_pool='avg')
        for p in self.cnn.parameters(): p.requires_grad = False
        for p in self.tfm.parameters(): p.requires_grad = False
        d = 256
        self.proj_cnn = nn.Sequential(nn.Linear(self.cnn.num_features, d), nn.GELU(), nn.LayerNorm(d))
        self.proj_tfm = nn.Sequential(nn.Linear(self.tfm.num_features, d), nn.GELU(), nn.LayerNorm(d))
        self.cross_attn = nn.MultiheadAttention(d, num_heads=4, batch_first=True)
        self.gate = nn.Sequential(nn.Linear(d*2, d), nn.GELU(),
                                  nn.Linear(d, 2), nn.Softmax(dim=-1))
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(d, num_classes))
    def forward(self, x, return_gate=False):
        with torch.no_grad():
            f_cnn = self.cnn(x); f_tfm = self.tfm(x)
        z_cnn = self.proj_cnn(f_cnn); z_tfm = self.proj_tfm(f_tfm)
        seq = torch.stack([z_cnn, z_tfm], dim=1)
        attn_out, _ = self.cross_attn(seq, seq, seq)
        g = self.gate(torch.cat([z_cnn, z_tfm], dim=-1))
        fused = g[:, 0:1] * attn_out[:, 0] + g[:, 1:2] * attn_out[:, 1]
        return (self.classifier(fused), g) if return_gate else self.classifier(fused)


class MSCA_Net(nn.Module):
    # Multi-Scale Cascade Attention
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = timm.create_model('convnext_tiny', pretrained=True,
                                          features_only=True, out_indices=(1, 2, 3))
        for p in self.backbone.parameters(): p.requires_grad = False
        chs = self.backbone.feature_info.channels(); d = 256
        self.proj = nn.ModuleList([nn.Sequential(nn.Conv2d(c, d, 1), nn.BatchNorm2d(d), nn.GELU())
                                   for c in chs])
        self.ca = nn.ModuleList([nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(d, d//8, 1), nn.ReLU(inplace=True),
            nn.Conv2d(d//8, d, 1), nn.Sigmoid()) for _ in chs])
        self.spatial_attn = nn.ModuleList([nn.Sequential(nn.Conv2d(d, 1, 1), nn.Sigmoid())
                                           for _ in chs])
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.scale_fusion = nn.Sequential(nn.Linear(d*len(chs), d), nn.GELU(), nn.LayerNorm(d))
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(d, num_classes))
    def forward(self, x):
        with torch.no_grad():
            feats_ = self.backbone(x)
        proj = [p(f) for p, f in zip(self.proj, feats_)]
        ref = [None]*len(proj)
        ref[-1] = proj[-1] * self.ca[-1](proj[-1])
        for i in range(len(proj)-2, -1, -1):
            a = self.spatial_attn[i+1](ref[i+1])
            a = F.interpolate(a, size=proj[i].shape[2:], mode='bilinear', align_corners=False)
            ref[i] = proj[i] * self.ca[i](proj[i]) * a
        pooled = torch.cat([self.gap(r).flatten(1) for r in ref], dim=1)
        return self.classifier(self.scale_fusion(pooled))

In [ ]:
def build_vgg16(num_classes=NUM_CLASSES):
    m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    for p in m.features.parameters(): p.requires_grad = False
    m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    return m

def build_resnet50(num_classes=NUM_CLASSES):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    for p in m.parameters(): p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

def build_densenet121(num_classes=NUM_CLASSES):
    return timm.create_model('densenet121', pretrained=True, num_classes=num_classes)
def build_mobilenetv3(num_classes=NUM_CLASSES):
    return timm.create_model('mobilenetv3_small_100', pretrained=True, num_classes=num_classes)
def build_efficientnet_b0(num_classes=NUM_CLASSES):
    return timm.create_model('efficientnet_b0', pretrained=True, num_classes=num_classes)
def build_efficientnet_b3(num_classes=NUM_CLASSES):
    return timm.create_model('tf_efficientnet_b3', pretrained=True, num_classes=num_classes)
def build_convnext(num_classes=NUM_CLASSES):
    return timm.create_model('convnext_tiny', pretrained=True, num_classes=num_classes)
def build_vit(num_classes=NUM_CLASSES):
    return timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes)
def build_swin(num_classes=NUM_CLASSES):
    return timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=num_classes)
def build_fspan_y(num_classes=NUM_CLASSES): return FSPAN_Y(num_classes)
def build_dpct(num_classes=NUM_CLASSES):    return DPCT_Net(num_classes)
def build_msca(num_classes=NUM_CLASSES):    return MSCA_Net(num_classes)


MODELS = {
    "VGG16":           build_vgg16,
    "ResNet50":        build_resnet50,
    "DenseNet121":     build_densenet121,
    "MobileNetV3":     build_mobilenetv3,
    "EfficientNet_B0": build_efficientnet_b0,
    "EfficientNet_B3": build_efficientnet_b3,
    "ConvNeXt_Tiny":   build_convnext,
    "ViT_Base":        build_vit,
    "Swin_Tiny":       build_swin,
    "FSPAN_Y":         build_fspan_y,
    "DPCT_Net":        build_dpct,
    "MSCA_Net":        build_msca,
}

MODEL_FAMILY = {
    "VGG16": "Classic CNN", "ResNet50": "Classic CNN", "DenseNet121": "Classic CNN",
    "MobileNetV3": "Efficient CNN", "EfficientNet_B0": "Efficient CNN",
    "EfficientNet_B3": "Efficient CNN", "ConvNeXt_Tiny": "Modern CNN",
    "ViT_Base": "Transformer", "Swin_Tiny": "Transformer",
    "FSPAN_Y": "Hybrid (existing)", "DPCT_Net": "Hybrid (proposed)",
    "MSCA_Net": "Hybrid (proposed)",
}

# frozen-backbone models (important for the reviewer's fairness concern)
FROZEN_BACKBONE = {"VGG16", "ResNet50", "FSPAN_Y", "DPCT_Net", "MSCA_Net"}

print(f"Registered {len(MODELS)} models:")
for n in MODELS:
    tag = "frozen" if n in FROZEN_BACKBONE else "fine-tuned"
    print(f"  {n:18s} {MODEL_FAMILY[n]:20s} [{tag}]")

In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

rows = []
for name, fn in MODELS.items():
    m = fn()
    tot, tr = count_parameters(m)
    rows.append({"Model": name, "Family": MODEL_FAMILY[name],
                 "Regime": "frozen" if name in FROZEN_BACKBONE else "fine-tuned",
                 "Total (M)": round(tot/1e6, 2), "Trainable (M)": round(tr/1e6, 3),
                 "Trainable %": round(100*tr/tot, 2)})
    del m
param_df = pd.DataFrame(rows)
param_df.to_csv(OUT_DIR / "model_parameters.csv", index=False)
print(param_df.to_string(index=False))

---
# 6. Training and evaluation utilities

One trainer used for every protocol; `tag` selects the output directory so
protocols never overwrite each other.

In [ ]:
import gc

def train_model(model, name, train_loader, val_loader, out_dir, tag,
                epochs=EPOCHS, force=False):
    wpath = out_dir / "models"  / f"{name}_{tag}_best.pth"
    hpath = out_dir / "metrics" / f"{name}_{tag}_history.json"

    if wpath.exists() and hpath.exists() and not force:
        print(f"  {name}: cached — loading")
        model.load_state_dict(torch.load(wpath, map_location=device, weights_only=True))
        return model.to(device), json.load(open(hpath))

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad],
                            lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)

    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc, best_state = 0.0, None

    for ep in range(epochs):
        model.train(); rl = rc = n = 0
        for x, y in tqdm(train_loader, desc=f"  ep{ep+1}/{epochs}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)          # frees grad buffers
            out = model(x); loss = criterion(out, y)
            loss.backward(); optimizer.step()
            rl += loss.item()*x.size(0); rc += (out.argmax(1) == y).sum().item(); n += x.size(0)
        tr_loss, tr_acc = rl/n, rc/n

        model.eval(); vl = vc = vn = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                out = model(x)
                vl += criterion(out, y).item()*x.size(0)
                vc += (out.argmax(1) == y).sum().item(); vn += x.size(0)
        va_loss, va_acc = vl/vn, vc/vn

        scheduler.step()
        hist["train_loss"].append(tr_loss); hist["train_acc"].append(tr_acc)
        hist["val_loss"].append(va_loss);   hist["val_acc"].append(va_acc)
        print(f"  ep{ep+1:2d}: train {tr_acc:.4f} | val {va_acc:.4f}")

        if va_acc > best_acc:
            best_acc = va_acc
            # keep the checkpoint on CPU, not GPU
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    hist["best_val_acc"] = best_acc
    torch.save(model.state_dict(), wpath)
    json.dump(hist, open(hpath, "w"), indent=2)
    print(f"  best val acc = {best_acc:.4f}")

    del optimizer, scheduler, criterion, best_state
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return model, hist

print("train_model() patched for memory")

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, name, n_classes=NUM_CLASSES):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    t0 = time.time(); n_img = 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        y_true.extend(y.numpy())
        y_pred.extend(logits.argmax(1).cpu().numpy())
        y_prob.extend(probs.cpu().numpy())
        n_img += x.size(0)
    elapsed = time.time() - t0

    y_true = np.array(y_true); y_pred = np.array(y_pred); y_prob = np.array(y_prob)
    labels = list(range(n_classes))
    m = {
        "accuracy":           accuracy_score(y_true, y_pred),
        "precision_macro":    precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro":       recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro":           f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted":    recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted":        f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "cohen_kappa":        cohen_kappa_score(y_true, y_pred),
        "mcc":                matthews_corrcoef(y_true, y_pred),
        "confusion_matrix":   confusion_matrix(y_true, y_pred, labels=labels).tolist(),
        "y_true": y_true.tolist(), "y_pred": y_pred.tolist(),
        "y_prob": y_prob.tolist(),
        "mean_inference_ms":  1000.0 * elapsed / max(n_img, 1),
    }
    # per-class and macro AUC
    try:
        yb = label_binarize(y_true, classes=labels)
        aucs = []
        for c in range(n_classes):
            if yb[:, c].sum() == 0: continue
            fpr, tpr, _ = roc_curve(yb[:, c], y_prob[:, c])
            aucs.append(auc(fpr, tpr))
        m["auc_macro"] = float(np.mean(aucs)) if aucs else float("nan")
    except Exception:
        m["auc_macro"] = float("nan")
    return m

print("train_model() and evaluate_model() ready")

---
# 7. Train on the original image-level split (Protocol A)

This reproduces the results in the submitted manuscript. If you already have
weights in `OUT_DIR/models`, they will be reused — nothing is retrained.

**Note.** Existing checkpoints from earlier runs are named `{name}_best.pth`.
The cell below looks for those first, so your previous training is not wasted.

In [ ]:
# migrate legacy checkpoint names ({name}_best.pth -> {name}_orig_best.pth)
import shutil
for name in MODELS:
    legacy = OUT_DIR / "models" / f"{name}_best.pth"
    newp   = OUT_DIR / "models" / f"{name}_orig_best.pth"
    lh     = OUT_DIR / "metrics" / f"{name}_history.json"
    nh     = OUT_DIR / "metrics" / f"{name}_orig_history.json"
    if legacy.exists() and not newp.exists():
        shutil.copy2(legacy, newp); print(f"  migrated weights: {name}")
    if lh.exists() and not nh.exists():
        shutil.copy2(lh, nh)
print("Legacy checkpoint migration done")

In [ ]:
all_metrics, all_histories = {}, {}

for name, fn in MODELS.items():
    print(f"\n{'='*70}\n  {name}  [Protocol A: image-level split]\n{'='*70}")
    model = fn(num_classes=NUM_CLASSES)
    model, hist = train_model(model, name, or_train_loader, or_val_loader,
                              OUT_DIR, "orig", force=FORCE_RETRAIN_ORIG)
    all_histories[name] = hist
    all_metrics[name] = evaluate_model(model, or_test_loader, name)
    print(f"  TEST acc={all_metrics[name]['accuracy']:.4f} "
          f"f1={all_metrics[name]['f1_macro']:.4f} "
          f"kappa={all_metrics[name]['cohen_kappa']:.4f}")
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"\nProtocol A complete: {len(all_metrics)} models")

In [ ]:
# ── Repair: evaluate any models missing from Protocol A ──
import traceback

missing = [n for n in MODELS if n not in all_metrics]
print(f"all_metrics: {len(all_metrics)}/12   missing: {missing}")

for name in missing:
    print(f"\n{'='*70}\n  {name}  [Protocol A repair]\n{'='*70}")
    try:
        model = MODELS[name](num_classes=NUM_CLASSES)
        model, hist = train_model(model, name, or_train_loader, or_val_loader,
                                  OUT_DIR, "orig", force=False)
        all_histories[name] = hist
        all_metrics[name]   = evaluate_model(model, or_test_loader, name)
        print(f"  TEST acc={all_metrics[name]['accuracy']:.4f} "
              f"f1={all_metrics[name]['f1_macro']:.4f}")
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception:
        print(f"  ERROR on {name}:"); traceback.print_exc()

print(f"\nall_metrics now: {len(all_metrics)}/12")

In [ ]:
# ── Persist metrics so a kernel restart doesn't lose them ──
def save_metrics(d, path):
    slim = {n: {k: v for k, v in m.items()
                if isinstance(v, (int, float, list, str))}
            for n, m in d.items()}
    json.dump(slim, open(path, "w"), indent=2)
    print(f"Saved {len(slim)} models -> {path}")

def load_metrics(path):
    if Path(path).exists():
        d = json.load(open(path))
        print(f"Loaded {len(d)} models <- {path}")
        return d
    print(f"Not found: {path}")
    return {}

save_metrics(all_metrics, OUT_DIR / "metrics" / "all_metrics_protocolA.json")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Corrected Table 5 — honest regime labels + efficiency framing
# Runs safely even if only Protocol A has finished.
# ══════════════════════════════════════════════════════════════════
TRAINING_REGIME = {
    "ResNet50":        "Linear probe (frozen backbone + new FC)",
    "VGG16":           "Conv frozen, FC classifier trained",
    "FSPAN_Y":         "Adapter head on frozen backbone",
    "DPCT_Net":        "Adapter head on frozen dual backbones",
    "MSCA_Net":        "Adapter head on frozen backbone",
    "DenseNet121":     "Full fine-tune",   "MobileNetV3":     "Full fine-tune",
    "EfficientNet_B0": "Full fine-tune",   "EfficientNet_B3": "Full fine-tune",
    "ConvNeXt_Tiny":   "Full fine-tune",   "ViT_Base":        "Full fine-tune",
    "Swin_Tiny":       "Full fine-tune",
}

# restore from disk if the kernel was restarted
if "all_metrics" not in dir() or len(all_metrics) == 0:
    all_metrics = load_metrics(OUT_DIR / "metrics" / "all_metrics_protocolA.json")
if "lr_metrics" not in dir() or len(lr_metrics) == 0:
    lr_metrics = load_metrics(LR_OUT_DIR / "metrics" / "all_metrics_protocolB.json")

def _get(d, n, key):
    return round(d[n][key], 4) if (d and n in d and key in d[n]) else np.nan

tbl = param_df.copy()
tbl["Regime"]   = tbl["Model"].map(TRAINING_REGIME)
tbl["Acc (A)"]  = tbl["Model"].map(lambda n: _get(all_metrics, n, "accuracy"))
tbl["Acc (B)"]  = tbl["Model"].map(lambda n: _get(lr_metrics,  n, "accuracy"))
# timing: prefer Protocol B, fall back to Protocol A
tbl["ms/img"]   = tbl["Model"].map(
    lambda n: _get(lr_metrics, n, "mean_inference_ms")
              if (lr_metrics and n in lr_metrics)
              else _get(all_metrics, n, "mean_inference_ms"))

tbl = tbl[["Model","Family","Regime","Total (M)","Trainable (M)","Trainable %",
           "ms/img","Acc (A)","Acc (B)"]].sort_values("Trainable (M)")
tbl.to_csv(OUT_DIR / "table5_efficiency_corrected.csv", index=False)
print(tbl.to_string(index=False))

n_a = int(tbl["Acc (A)"].notna().sum()); n_b = int(tbl["Acc (B)"].notna().sum())
print(f"\nProtocol A: {n_a}/12 models   Protocol B: {n_b}/12 models")
if n_b == 0:
    print("Protocol B is empty — run Section 8, then re-run this cell.")

---
# 8. Train on the leakage-resistant split (Protocol B)

Same protocol, group-aware split. **This is the run reviewers asked for.**

In [ ]:
lr_metrics, lr_histories = {}, {}

for name, fn in MODELS.items():
    print(f"\n{'='*70}\n  {name}  [Protocol B: leakage-resistant]\n{'='*70}")
    model = fn(num_classes=NUM_CLASSES)
    model, hist = train_model(model, name, lr_train_loader, lr_val_loader,
                              LR_OUT_DIR, "LR", force=FORCE_RETRAIN_LR)
    lr_histories[name] = hist
    lr_metrics[name] = evaluate_model(model, lr_test_loader, name)
    print(f"  TEST acc={lr_metrics[name]['accuracy']:.4f} "
          f"f1={lr_metrics[name]['f1_macro']:.4f} "
          f"kappa={lr_metrics[name]['cohen_kappa']:.4f}")
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"\nProtocol B complete: {len(lr_metrics)} models")

In [ ]:
save_metrics(lr_metrics, LR_OUT_DIR / "metrics" / "all_metrics_protocolB.json")

---
# 9. Dual-protocol comparison — the headline result

In [ ]:
# Restore metrics if the kernel was restarted
if "all_metrics" not in dir() or len(all_metrics) == 0:
    all_metrics = load_metrics(OUT_DIR / "metrics" / "all_metrics_protocolA.json")
if "lr_metrics" not in dir() or len(lr_metrics) == 0:
    lr_metrics = load_metrics(LR_OUT_DIR / "metrics" / "all_metrics_protocolB.json")

In [ ]:
rows = []
for name in MODELS:
    if name not in all_metrics or name not in lr_metrics: continue
    a, b = all_metrics[name]["accuracy"], lr_metrics[name]["accuracy"]
    rows.append({
        "Model": name, "Family": MODEL_FAMILY[name],
        "Regime": "frozen" if name in FROZEN_BACKBONE else "fine-tuned",
        "Acc (image-level)": round(a, 4),
        "Acc (leakage-resistant)": round(b, 4),
        "Drop (pp)": round((a-b)*100, 2),
        "F1 (LR)": round(lr_metrics[name]["f1_macro"], 4),
        "Kappa (LR)": round(lr_metrics[name]["cohen_kappa"], 4),
        "MCC (LR)": round(lr_metrics[name]["mcc"], 4),
    })
dual_df = pd.DataFrame(rows).sort_values("Acc (leakage-resistant)", ascending=False)
dual_df.to_csv(LR_OUT_DIR / "dual_protocol_comparison.csv", index=False)
print(dual_df.to_string(index=False))
print(f"\nMean drop   : {dual_df['Drop (pp)'].mean():.2f} pp")
print(f"Largest drop: {dual_df.iloc[dual_df['Drop (pp)'].argmax()]['Model']} "
      f"({dual_df['Drop (pp)'].max():.2f} pp)")
print(f"Smallest    : {dual_df.iloc[dual_df['Drop (pp)'].argmin()]['Model']} "
      f"({dual_df['Drop (pp)'].min():.2f} pp)")
n_ceiling_a = (dual_df["Acc (image-level)"] > 0.998).sum()
n_ceiling_b = (dual_df["Acc (leakage-resistant)"] > 0.998).sum()
print(f"\nModels above 99.8%: image-level {n_ceiling_a}/12 -> leakage-resistant {n_ceiling_b}/12")

In [ ]:
names = list(dual_df["Model"]); x = np.arange(len(names)); w = 0.38
fig, ax = plt.subplots(figsize=(max(11, 1.1*len(names)), 5.5))
ax.bar(x - w/2, dual_df["Acc (image-level)"], w,
       label="Image-level split (leaky)", color="#C0392B", alpha=0.85)
ax.bar(x + w/2, dual_df["Acc (leakage-resistant)"], w,
       label="Leakage-resistant split", color="#1C7293", alpha=0.85)
for i, (a, b) in enumerate(zip(dual_df["Acc (image-level)"], dual_df["Acc (leakage-resistant)"])):
    ax.text(i - w/2, a + 0.006, f"{a:.3f}", ha="center", fontsize=7.5)
    ax.text(i + w/2, b + 0.006, f"{b:.3f}", ha="center", fontsize=7.5)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=25, ha="right")
ax.set_ylabel("Test accuracy"); ax.set_ylim(0, 1.08)
ax.set_title("LC25000 accuracy under both splitting protocols")
ax.legend(loc="lower left"); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(LR_OUT_DIR / "figures" / "dual_protocol_comparison.png", dpi=600, bbox_inches='tight')
plt.show()

---
# 10. Full metrics, confusion matrices, ROC, per-class

Set `ACTIVE` below to choose which protocol to analyse.

In [ ]:
# Choose the protocol for all analysis below
ACTIVE_METRICS = lr_metrics        # or: all_metrics
ACTIVE_LABEL   = "leakage-resistant"
ACTIVE_OUT     = LR_OUT_DIR
print(f"Analysing protocol: {ACTIVE_LABEL}  ({len(ACTIVE_METRICS)} models)")

In [ ]:
rows = []
for name, m in ACTIVE_METRICS.items():
    rows.append({"Model": name, "Family": MODEL_FAMILY[name],
                 "Accuracy": m["accuracy"], "Precision": m["precision_macro"],
                 "Recall": m["recall_macro"], "F1": m["f1_macro"],
                 "Kappa": m["cohen_kappa"], "MCC": m["mcc"],
                 "AUC": m.get("auc_macro", float("nan")),
                 "Inference (ms)": m["mean_inference_ms"]})
results_df = pd.DataFrame(rows).sort_values("Accuracy", ascending=False).round(4)
results_df.to_csv(ACTIVE_OUT / "results_summary.csv", index=False)
print(results_df.to_string(index=False))

In [ ]:
n = len(ACTIVE_METRICS); ncol = 4; nrow = int(np.ceil(n/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2*ncol, 3.6*nrow))
axes = np.atleast_1d(axes).ravel()
for ax, (name, m) in zip(axes, ACTIVE_METRICS.items()):
    cm = np.array(m["confusion_matrix"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f"{name}  ({m['accuracy']:.4f})", fontsize=10)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.setp(ax.get_xticklabels(), rotation=40, ha="right", fontsize=7)
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=7)
for ax in axes[n:]: ax.axis("off")
plt.suptitle(f"Confusion matrices — {ACTIVE_LABEL}", y=1.002, fontsize=13)
plt.tight_layout()
plt.savefig(ACTIVE_OUT / "figures" / "confusion_matrices.png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# per-class precision / recall / F1 heatmaps
metrics3 = ["precision", "recall", "f1"]
grids = {k: np.zeros((len(ACTIVE_METRICS), NUM_CLASSES)) for k in metrics3}
model_names = list(ACTIVE_METRICS.keys())
for r, name in enumerate(model_names):
    yt = np.array(ACTIVE_METRICS[name]["y_true"]); yp = np.array(ACTIVE_METRICS[name]["y_pred"])
    grids["precision"][r] = precision_score(yt, yp, average=None, labels=range(NUM_CLASSES), zero_division=0)
    grids["recall"][r]    = recall_score(yt, yp, average=None, labels=range(NUM_CLASSES), zero_division=0)
    grids["f1"][r]        = f1_score(yt, yp, average=None, labels=range(NUM_CLASSES), zero_division=0)

fig, axes = plt.subplots(1, 3, figsize=(20, max(4, 0.42*len(model_names))))
for ax, k in zip(axes, metrics3):
    sns.heatmap(grids[k], annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.5, vmax=1.0,
                xticklabels=CLASS_NAMES, yticklabels=model_names, ax=ax, cbar=True)
    ax.set_title(k.capitalize(), fontsize=12)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)
plt.suptitle(f"Per-class metrics — {ACTIVE_LABEL}", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(ACTIVE_OUT / "figures" / "per_class_heatmaps.png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# ROC curves
fig, axes = plt.subplots(nrow, ncol, figsize=(4.0*ncol, 3.4*nrow))
axes = np.atleast_1d(axes).ravel()
for ax, (name, m) in zip(axes, ACTIVE_METRICS.items()):
    yt = np.array(m["y_true"]); yp = np.array(m["y_prob"])
    yb = label_binarize(yt, classes=list(range(NUM_CLASSES)))
    for c in range(NUM_CLASSES):
        if yb[:, c].sum() == 0: continue
        fpr, tpr, _ = roc_curve(yb[:, c], yp[:, c])
        ax.plot(fpr, tpr, lw=1.2, label=f"{CLASS_NAMES[c]} ({auc(fpr,tpr):.3f})")
    ax.plot([0,1], [0,1], 'k--', lw=0.7)
    ax.set_title(f"{name}\nmacro AUC = {m.get('auc_macro', float('nan')):.4f}", fontsize=9)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.legend(fontsize=5.5, loc="lower right")
for ax in axes[n:]: ax.axis("off")
plt.suptitle(f"ROC curves (one-vs-rest) — {ACTIVE_LABEL}", y=1.002, fontsize=13)
plt.tight_layout()
plt.savefig(ACTIVE_OUT / "figures" / "roc_curves.png", dpi=600, bbox_inches='tight')
plt.show()

---
# 11. TP / FP / TN / FN analysis

One-vs-rest decomposition of each confusion matrix (committee request).

In [ ]:
def confusion_components(cm):
    cm = np.asarray(cm); total = int(cm.sum()); out = {}
    for c in range(cm.shape[0]):
        TP = int(cm[c, c]); FP = int(cm[:, c].sum() - TP)
        FN = int(cm[c, :].sum() - TP); TN = total - TP - FP - FN
        out[c] = {"TP": TP, "FP": FP, "TN": TN, "FN": FN,
                  "Sensitivity": TP/(TP+FN) if TP+FN else 0.0,
                  "Specificity": TN/(TN+FP) if TN+FP else 0.0,
                  "PPV":         TP/(TP+FP) if TP+FP else 0.0}
    return out


rows = []
for name, m in ACTIVE_METRICS.items():
    for c, v in confusion_components(m["confusion_matrix"]).items():
        rows.append({"Model": name, "Class": CLASS_NAMES[c],
                     "TP": v["TP"], "FP": v["FP"], "TN": v["TN"], "FN": v["FN"],
                     "Sensitivity": round(v["Sensitivity"], 4),
                     "Specificity": round(v["Specificity"], 4),
                     "PPV": round(v["PPV"], 4)})
tpfp_df = pd.DataFrame(rows)
tpfp_df.to_csv(ACTIVE_OUT / "tp_fp_tn_fn_per_class.csv", index=False)
print(tpfp_df.head(15).to_string(index=False))
print(f"\nSaved {len(tpfp_df)} rows")

---
# 12. Stain-robustness evaluation

**Terminology note for the manuscript:** this is *synthetic colour perturbation*,
not real inter-laboratory stain variation. Reviewers R2-3, R3-5 and R6-1 asked
for this to be stated precisely.

In [ ]:
stain_rows = []
for name, fn in MODELS.items():
    wpath = LR_OUT_DIR / "models" / f"{name}_LR_best.pth"
    if not wpath.exists():
        print(f"  skip {name}: no leakage-resistant weights"); continue
    model = fn(num_classes=NUM_CLASSES)
    model.load_state_dict(torch.load(wpath, map_location=device, weights_only=True))
    model.to(device).eval()
    m_clean = ACTIVE_METRICS[name]["accuracy"]
    m_pert  = evaluate_model(model, lr_stain_loader, name)["accuracy"]
    stain_rows.append({"Model": name, "Family": MODEL_FAMILY[name],
                       "Clean": round(m_clean, 4), "Perturbed": round(m_pert, 4),
                       "Drop (pp)": round((m_clean-m_pert)*100, 2),
                       "Relative drop (%)": round(100*(m_clean-m_pert)/max(m_clean,1e-9), 2)})
    print(f"  {name:18s} clean {m_clean:.4f} -> perturbed {m_pert:.4f}")
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

stain_df = pd.DataFrame(stain_rows).sort_values("Perturbed", ascending=False)
stain_df.to_csv(ACTIVE_OUT / "stain_robustness.csv", index=False)
print()
print(stain_df.to_string(index=False))

In [ ]:
x = np.arange(len(stain_df)); w = 0.38
fig, ax = plt.subplots(figsize=(max(11, 1.1*len(stain_df)), 5))
ax.bar(x - w/2, stain_df["Clean"], w, label="Clean test", color="#2E8B57", alpha=0.85)
ax.bar(x + w/2, stain_df["Perturbed"], w, label="Colour-perturbed test", color="#D2691E", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(stain_df["Model"], rotation=25, ha="right")
ax.set_ylabel("Accuracy"); ax.set_ylim(0, 1.08)
ax.set_title(f"Synthetic colour-perturbation robustness — {ACTIVE_LABEL}")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(ACTIVE_OUT / "figures" / "stain_robustness.png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
x = np.arange(len(stain_df)); w = 0.38
fig, ax = plt.subplots(figsize=(max(11, 1.1*len(stain_df)), 5))

# 1. Assign the bars to variables
bars1 = ax.bar(x - w/2, stain_df["Clean"], w, label="Clean test", color="#2E8B57", alpha=0.85)
bars2 = ax.bar(x + w/2, stain_df["Perturbed"], w, label="Colour-perturbed test", color="#D2691E", alpha=0.85)

# 2. Add the vertical text labels inside the bars
# label_type='center' centers it in the bar. fmt='%.4f' keeps it to 4 decimal places to match your printout.
ax.bar_label(bars1, fmt='%.4f', label_type='center', rotation=90, color='white', fontsize=9)
ax.bar_label(bars2, fmt='%.4f', label_type='center', rotation=90, color='white', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(stain_df["Model"], rotation=25, ha="right")
ax.set_ylabel("Accuracy"); ax.set_ylim(0, 1.08)
ax.set_title(f"Synthetic colour-perturbation robustness - {ACTIVE_LABEL}")

# 3. Move the legend to the bottom left
ax.legend(loc="lower left")

ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(ACTIVE_OUT / "figures" / "stain_robustness1.png", dpi=600, bbox_inches='tight')
plt.show()

---
# 13. Efficiency analysis

Reports **total** parameters alongside trainable, addressing R4-7, R5-6, R6-10:
trainable count is not deployment cost.

In [ ]:
eff = param_df.copy()
eff["Accuracy"] = eff["Model"].map(lambda n: ACTIVE_METRICS[n]["accuracy"] if n in ACTIVE_METRICS else np.nan)
eff["Inference (ms)"] = eff["Model"].map(lambda n: ACTIVE_METRICS[n]["mean_inference_ms"] if n in ACTIVE_METRICS else np.nan)
if len(stain_df):
    eff["Perturbed Acc"] = eff["Model"].map(dict(zip(stain_df["Model"], stain_df["Perturbed"])))
eff = eff.dropna(subset=["Accuracy"]).sort_values("Accuracy", ascending=False)
eff.to_csv(ACTIVE_OUT / "efficiency_summary.csv", index=False)
print(eff.to_string(index=False))

In [ ]:
ycol = "Perturbed Acc" if "Perturbed Acc" in eff.columns else "Accuracy"
fig, ax = plt.subplots(figsize=(10, 6.5))
fam_colors = {"Classic CNN":"#E07B54","Efficient CNN":"#F5C242","Modern CNN":"#4A90B8",
              "Transformer":"#7B68EE","Hybrid (existing)":"#3DBD91","Hybrid (proposed)":"#C0392B"}

for _, r in eff.iterrows():
    # Only draw the bubble, no annotations
    ax.scatter(r["Total (M)"], r[ycol], s=60 + 22*r["Inference (ms)"],
               color=fam_colors.get(r["Family"], "#888"), alpha=0.8,
               edgecolors="white", linewidths=1.2, zorder=3)

ax.set_xscale("log")
ax.set_xlabel("Total parameters (millions, log scale)")
ax.set_ylabel(ycol)
ax.set_title("Efficiency — bubble size proportional to inference time")

handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=c,
                      markersize=9, label=f) for f, c in fam_colors.items()]
ax.legend(handles=handles, fontsize=8, loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()

# Removed the '#' to activate saving
plt.savefig(ACTIVE_OUT / "figures" / "efficiency_pareto.png", dpi=600, bbox_inches='tight')
plt.show()

---
# 14. LungHist700 — external dataset

Two experiments:
1. **Cross-dataset transfer** — LC25000-trained models evaluated directly.
2. **Fine-tuned** — trained on LungHist700 as an independent 3-class task.

Per R3-2 this must be described as *external-dataset transfer/fine-tuning*,
not zero-shot generalisation.

In [ ]:
CLASS_NAMES_3 = ['lung_aca', 'lung_n', 'lung_scc']
LH_TO_LC = {'aca': CLASS_NAMES.index('lung_aca'),
            'nor': CLASS_NAMES.index('lung_n'),
            'ssc': CLASS_NAMES.index('lung_scc')}

class LungHist700(Dataset):
    def __init__(self, root, folder_to_idx, transform=None):
        self.root = Path(root); self.transform = transform; self.samples = []
        for folder, idx in folder_to_idx.items():
            d = self.root / folder
            if not d.exists(): continue
            for p in sorted(d.iterdir()):
                if p.suffix.lower() in IMG_EXTS:
                    self.samples.append((p, idx))
        if not self.samples:
            raise RuntimeError(f"No images under {self.root}")
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, y = self.samples[i]
        img = Image.open(p).convert("RGB")
        if self.transform is not None: img = self.transform(img)
        return img, y

lh_ds = LungHist700(LUNGHIST_DIR, LH_TO_LC, eval_transform)
lh_loader = make_loader(lh_ds, False)
cnt = Counter(y for _, y in lh_ds.samples)
print(f"LungHist700: {len(lh_ds)} images")
for f, i in LH_TO_LC.items():
    print(f"  {f} -> {CLASS_NAMES[i]}: {cnt[i]}")

In [ ]:
# 14.1 Cross-dataset transfer (no retraining)
xd_rows = []
colon_idx = {CLASS_NAMES.index('colon_aca'), CLASS_NAMES.index('colon_n')}
for name, fn in MODELS.items():
    wpath = LR_OUT_DIR / "models" / f"{name}_LR_best.pth"
    if not wpath.exists(): continue
    model = fn(num_classes=NUM_CLASSES)
    model.load_state_dict(torch.load(wpath, map_location=device, weights_only=True))
    model.to(device).eval()
    yt, yp = [], []
    with torch.no_grad():
        for x, y in lh_loader:
            yp.extend(model(x.to(device)).argmax(1).cpu().numpy()); yt.extend(y.numpy())
    yt, yp = np.array(yt), np.array(yp)
    ood = int(sum(1 for p in yp if p in colon_idx))
    xd_rows.append({"Model": name, "Accuracy": round(float((yt == yp).mean()), 4),
                    "OOD colon preds": ood,
                    "OOD rate (%)": round(100*ood/len(yt), 2)})
    print(f"  {name:18s} acc={xd_rows[-1]['Accuracy']:.4f}  OOD={ood}")
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

xd_df = pd.DataFrame(xd_rows).sort_values("Accuracy", ascending=False)
xd_df.to_csv(LH_OUT_DIR / "cross_dataset_results.csv", index=False)
print(); print(xd_df.to_string(index=False))

In [ ]:
# 14.2 Fine-tuned on LungHist700 (3-class)
# NOTE: LungHist700 has 691 images from 45 patients. Patient IDs are not consistently exposed, so this is a stratified per-image split — state this limitation in the manuscript.
from sklearn.model_selection import train_test_split as tts

lh_eval = LungHist700(LUNGHIST_DIR, LH_TO_LC, eval_transform)
lh_aug  = LungHist700(LUNGHIST_DIR, LH_TO_LC, train_transform)
labels_lh = np.array([y for _, y in lh_eval.samples])
idx_lh = np.arange(len(lh_eval))

tr_lh, tmp_lh = tts(idx_lh, test_size=0.30, random_state=SEED, stratify=labels_lh)
va_lh, te_lh  = tts(tmp_lh, test_size=0.50, random_state=SEED, stratify=labels_lh[tmp_lh])

LC_TO_3 = {CLASS_NAMES.index(c): i for i, c in enumerate(CLASS_NAMES_3)}
class Remap3(Dataset):
    def __init__(self, base, idx): self.base, self.idx = base, idx
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        x, y = self.base[self.idx[i]]
        return x, LC_TO_3[int(y)]

lh_tr_loader = make_loader(Remap3(lh_aug,  tr_lh), True)
lh_va_loader = make_loader(Remap3(lh_eval, va_lh), False)
lh_te_loader = make_loader(Remap3(lh_eval, te_lh), False)
print(f"LungHist700 split — train {len(tr_lh)} | val {len(va_lh)} | test {len(te_lh)}")

In [ ]:
lh_metrics = {}
for name, fn in MODELS.items():
    print(f"\n{'='*70}\n  {name}  [LungHist700 fine-tune]\n{'='*70}")
    model = fn(num_classes=3)
    model, _ = train_model(model, name, lh_tr_loader, lh_va_loader,
                           LH_OUT_DIR, "LH", force=FORCE_RETRAIN_LH)
    lh_metrics[name] = evaluate_model(model, lh_te_loader, name, n_classes=3)
    print(f"  TEST acc={lh_metrics[name]['accuracy']:.4f} "
          f"f1={lh_metrics[name]['f1_macro']:.4f}")
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

lh_df = pd.DataFrame({n: {k: round(v, 4) for k, v in m.items()
                          if isinstance(v, (int, float))}
                      for n, m in lh_metrics.items()}).T
lh_df.index.name = "Model"
lh_df = lh_df.sort_values("accuracy", ascending=False)
lh_df.to_csv(LH_OUT_DIR / "lunghist700_finetuned.csv")
print(); print(lh_df.to_string())

---
# 15. Statistical analysis

Addresses R4-5 and R6-8: full McNemar contingency counts with exact p-values,
plus Wilson confidence intervals on every accuracy.

In [ ]:
from scipy import stats as sstats

def wilson_ci(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return (max(0.0, c-h), min(1.0, c+h))

ci_rows = []
for name, m in ACTIVE_METRICS.items():
    yt = np.array(m["y_true"]); yp = np.array(m["y_pred"])
    k, n = int((yt == yp).sum()), len(yt)
    lo, hi = wilson_ci(k, n)
    ci_rows.append({"Model": name, "Correct": k, "N": n,
                    "Accuracy": round(k/n, 4),
                    "95% CI low": round(lo, 4), "95% CI high": round(hi, 4),
                    "CI width (pp)": round((hi-lo)*100, 2)})
ci_df = pd.DataFrame(ci_rows).sort_values("Accuracy", ascending=False)
ci_df.to_csv(ACTIVE_OUT / "accuracy_confidence_intervals.csv", index=False)
print(ci_df.to_string(index=False))

In [ ]:
# Pairwise McNemar with full contingency counts and exact p-values
def mcnemar_exact(y_true, p1, p2):
    c1 = (p1 == y_true); c2 = (p2 == y_true)
    b = int(np.sum(c1 & ~c2))   # model1 right, model2 wrong
    c = int(np.sum(~c1 & c2))   # model1 wrong, model2 right
    if b + c == 0: return b, c, 1.0
    p = sstats.binomtest(b, b+c, 0.5).pvalue     # exact binomial
    return b, c, float(p)

names_s = list(ACTIVE_METRICS.keys())
mc_rows = []
for i in range(len(names_s)):
    for j in range(i+1, len(names_s)):
        n1, n2 = names_s[i], names_s[j]
        yt = np.array(ACTIVE_METRICS[n1]["y_true"])
        b, c, p = mcnemar_exact(yt,
                                np.array(ACTIVE_METRICS[n1]["y_pred"]),
                                np.array(ACTIVE_METRICS[n2]["y_pred"]))
        mc_rows.append({"Model A": n1, "Model B": n2,
                        "b (A right, B wrong)": b, "c (A wrong, B right)": c,
                        "p-value": round(p, 6),
                        "Significant (a=0.05)": "yes" if p < 0.05 else "no"})
mc_df = pd.DataFrame(mc_rows).sort_values("p-value")
mc_df.to_csv(ACTIVE_OUT / "mcnemar_pairwise.csv", index=False)
print(f"{len(mc_df)} pairwise comparisons")
print(mc_df.head(20).to_string(index=False))
print(f"\nSignificant pairs: {(mc_df['p-value'] < 0.05).sum()} / {len(mc_df)}")

---
# 16. Export everything for the manuscript

In [ ]:
import zipfile

manifest = []
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file() and p.suffix.lower() in {".csv", ".png", ".json"}:
        manifest.append({"file": str(p.relative_to(OUT_DIR)),
                         "size_kb": round(p.stat().st_size/1024, 1)})
man_df = pd.DataFrame(manifest)
man_df.to_csv(OUT_DIR / "export_manifest.csv", index=False)
print(f"{len(man_df)} result files")
print(man_df.to_string(index=False))

In [ ]:
zip_path = OUT_DIR / "paper_package.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT_DIR.rglob("*")):
        if p.is_file() and p.suffix.lower() in {".csv", ".png"} and p != zip_path:
            z.write(p, p.relative_to(OUT_DIR))
print(f"Packaged -> {zip_path}  ({zip_path.stat().st_size/1e6:.1f} MB)")
print("\nPipeline complete.")

---
## Key numbers for the manuscript

Run this last cell for a one-screen summary of everything the reviewers asked about.

In [ ]:
print("="*72)
print("SUMMARY FOR THE MANUSCRIPT")
print("="*72)

print("\n[Leakage quantification]")
print(leak_df.to_string(index=False))
print(f"  Mean max-similarity: original {sims_old.mean():.4f} -> group-aware {sims_new.mean():.4f}")
print(f"  Groups recovered: {lc_df['group_final'].nunique()} (ground truth 1250)")
print(f"  Merge threshold: {CHOSEN_T}")

if len(dual_df):
    print("\n[Dual-protocol accuracy]")
    print(f"  Mean drop: {dual_df['Drop (pp)'].mean():.2f} pp")
    print(f"  Models >99.8%: {(dual_df['Acc (image-level)']>0.998).sum()}/12 "
          f"-> {(dual_df['Acc (leakage-resistant)']>0.998).sum()}/12")
    print(f"  Best (leakage-resistant): {dual_df.iloc[0]['Model']} "
          f"{dual_df.iloc[0]['Acc (leakage-resistant)']:.4f}")

if len(stain_df):
    print("\n[Colour-perturbation robustness]")
    print(f"  Most robust : {stain_df.iloc[0]['Model']} {stain_df.iloc[0]['Perturbed']:.4f}")
    print(f"  Least robust: {stain_df.iloc[-1]['Model']} {stain_df.iloc[-1]['Perturbed']:.4f}")

if len(xd_df):
    print("\n[LungHist700 cross-dataset]")
    print(f"  Best: {xd_df.iloc[0]['Model']} {xd_df.iloc[0]['Accuracy']:.4f}")

print("\n[Statistics]")
print(f"  Mean 95% CI width: {ci_df['CI width (pp)'].mean():.2f} pp")
print(f"  Significant McNemar pairs: {(mc_df['p-value']<0.05).sum()}/{len(mc_df)}")
print("="*72)

---
---
# PART II — Reviewer-Response Experiments

Sections 17–21 address the remaining reviewer requirements. All experiments run
on the **leakage-resistant split** (Protocol B).

| Section | Addresses |
|---|---|
| 17 | Ablation studies — R1-10, R4-4, R4-6, R6-4, R6-5 |
| 18 | Multi-seed / repeated splits — R2-4, R4-5, R5-5, R6-9 |
| 19 | Matched training regimes — R1-4/5/9, R2-2/5, R3-1, R4-2, R6-2 |
| 20 | Gating-weight distributions — R5-7, R6-6 |
| 21 | Grad-CAM at scale with quantified agreement — R5-7, R6-7 |

**Estimated runtime:** Section 17 ≈ 2 h, Section 18 ≈ 5 h, Section 19 ≈ 1 h,
Sections 20–21 ≈ 15 min. All are resume-aware.

---
# 17. Ablation studies

**DPCT-Net (7 variants)** isolates dual backbones, fusion type, cross-attention
and gating. **MSCA-Net (5 variants)** isolates multi-scale features, channel
attention, spatial attention and the top-down cascade.

The `concat` and `mean` variants directly answer R4-6 and R6-5: *is attention
between two global tokens better than simple fusion?*

In [ ]:
ABL_OUT_DIR = OUT_DIR / "ablations"
(ABL_OUT_DIR / "models").mkdir(parents=True, exist_ok=True)
(ABL_OUT_DIR / "metrics").mkdir(parents=True, exist_ok=True)
(ABL_OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)


class DPCT_Ablation(nn.Module):
    # fusion in {'concat','mean','gate','attn','gate_attn'}
    def __init__(self, num_classes=NUM_CLASSES, use_cnn=True, use_tfm=True,
                 fusion="gate_attn"):
        super().__init__()
        self.use_cnn, self.use_tfm, self.fusion = use_cnn, use_tfm, fusion
        d = 256
        if use_cnn:
            self.cnn = timm.create_model('convnext_tiny', pretrained=True,
                                         num_classes=0, global_pool='avg')
            for p in self.cnn.parameters(): p.requires_grad = False
            self.proj_cnn = nn.Sequential(nn.Linear(self.cnn.num_features, d),
                                          nn.GELU(), nn.LayerNorm(d))
        if use_tfm:
            self.tfm = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True,
                                         num_classes=0, global_pool='avg')
            for p in self.tfm.parameters(): p.requires_grad = False
            self.proj_tfm = nn.Sequential(nn.Linear(self.tfm.num_features, d),
                                          nn.GELU(), nn.LayerNorm(d))
        both = use_cnn and use_tfm
        if both and fusion in ("attn", "gate_attn"):
            self.cross_attn = nn.MultiheadAttention(d, num_heads=4, batch_first=True)
        if both and fusion in ("gate", "gate_attn"):
            self.gate = nn.Sequential(nn.Linear(d*2, d), nn.GELU(),
                                      nn.Linear(d, 2), nn.Softmax(dim=-1))
        in_dim = d*2 if (both and fusion == "concat") else d
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_dim, num_classes))

    def forward(self, x, return_gate=False):
        with torch.no_grad():
            f_c = self.cnn(x) if self.use_cnn else None
            f_t = self.tfm(x) if self.use_tfm else None
        zs = []
        if self.use_cnn: zs.append(self.proj_cnn(f_c))
        if self.use_tfm: zs.append(self.proj_tfm(f_t))

        g = None
        if len(zs) == 1:
            fused = zs[0]
        else:
            z_c, z_t = zs
            if self.fusion == "concat":
                fused = torch.cat([z_c, z_t], dim=-1)
            elif self.fusion == "mean":
                fused = 0.5 * (z_c + z_t)
            else:
                if self.fusion in ("attn", "gate_attn"):
                    seq = torch.stack([z_c, z_t], dim=1)
                    ao, _ = self.cross_attn(seq, seq, seq)
                    a_c, a_t = ao[:, 0], ao[:, 1]
                else:
                    a_c, a_t = z_c, z_t
                if self.fusion in ("gate", "gate_attn"):
                    g = self.gate(torch.cat([z_c, z_t], dim=-1))
                    fused = g[:, 0:1] * a_c + g[:, 1:2] * a_t
                else:
                    fused = 0.5 * (a_c + a_t)
        out = self.classifier(fused)
        return (out, g) if return_gate else out


DPCT_VARIANTS = {
    "A1_CNN_only":        dict(use_cnn=True,  use_tfm=False, fusion="gate_attn"),
    "A2_Swin_only":       dict(use_cnn=False, use_tfm=True,  fusion="gate_attn"),
    "A3_concat_MLP":      dict(use_cnn=True,  use_tfm=True,  fusion="concat"),
    "A4_mean_fusion":     dict(use_cnn=True,  use_tfm=True,  fusion="mean"),
    "A5_gate_only":       dict(use_cnn=True,  use_tfm=True,  fusion="gate"),
    "A6_crossattn_only":  dict(use_cnn=True,  use_tfm=True,  fusion="attn"),
    "A7_FULL_DPCT":       dict(use_cnn=True,  use_tfm=True,  fusion="gate_attn"),
}
print(f"DPCT-Net ablation: {len(DPCT_VARIANTS)} variants")
for k, v in DPCT_VARIANTS.items(): print(f"  {k:20s} {v}")

In [ ]:
class MSCA_Ablation(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, out_indices=(1, 2, 3),
                 use_ca=True, use_sa=True, cascade=True):
        super().__init__()
        self.use_ca, self.use_sa, self.cascade = use_ca, use_sa, cascade
        self.backbone = timm.create_model('convnext_tiny', pretrained=True,
                                          features_only=True, out_indices=out_indices)
        for p in self.backbone.parameters(): p.requires_grad = False
        chs = self.backbone.feature_info.channels(); d = 256
        self.proj = nn.ModuleList([nn.Sequential(nn.Conv2d(c, d, 1),
                                                 nn.BatchNorm2d(d), nn.GELU()) for c in chs])
        if use_ca:
            self.ca = nn.ModuleList([nn.Sequential(
                nn.AdaptiveAvgPool2d(1), nn.Conv2d(d, d//8, 1), nn.ReLU(inplace=True),
                nn.Conv2d(d//8, d, 1), nn.Sigmoid()) for _ in chs])
        if use_sa:
            self.sa = nn.ModuleList([nn.Sequential(nn.Conv2d(d, 1, 1), nn.Sigmoid())
                                     for _ in chs])
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.scale_fusion = nn.Sequential(nn.Linear(d*len(chs), d), nn.GELU(), nn.LayerNorm(d))
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(d, num_classes))

    def forward(self, x):
        with torch.no_grad():
            feats_ = self.backbone(x)
        proj = [p(f) for p, f in zip(self.proj, feats_)]
        n = len(proj)
        ref = [None]*n

        if self.cascade and self.use_sa and n > 1:
            ref[-1] = proj[-1] * self.ca[-1](proj[-1]) if self.use_ca else proj[-1]
            for i in range(n-2, -1, -1):
                a = self.sa[i+1](ref[i+1])
                a = F.interpolate(a, size=proj[i].shape[2:], mode='bilinear', align_corners=False)
                base = proj[i] * self.ca[i](proj[i]) if self.use_ca else proj[i]
                ref[i] = base * a
        else:
            for i in range(n):
                r = proj[i] * self.ca[i](proj[i]) if self.use_ca else proj[i]
                if self.use_sa:
                    r = r * self.sa[i](r)
                ref[i] = r

        pooled = torch.cat([self.gap(r).flatten(1) for r in ref], dim=1)
        return self.classifier(self.scale_fusion(pooled))


MSCA_VARIANTS = {
    "B1_deepest_only":     dict(out_indices=(3,),      use_ca=False, use_sa=False, cascade=False),
    "B2_multiscale_only":  dict(out_indices=(1, 2, 3), use_ca=False, use_sa=False, cascade=False),
    "B3_plus_channel_att": dict(out_indices=(1, 2, 3), use_ca=True,  use_sa=False, cascade=False),
    "B4_plus_spatial_ind": dict(out_indices=(1, 2, 3), use_ca=True,  use_sa=True,  cascade=False),
    "B5_FULL_MSCA":        dict(out_indices=(1, 2, 3), use_ca=True,  use_sa=True,  cascade=True),
}
print(f"MSCA-Net ablation: {len(MSCA_VARIANTS)} variants")
for k, v in MSCA_VARIANTS.items(): print(f"  {k:22s} {v}")

In [ ]:
def run_ablation(variants, cls, prefix):
    results = {}
    for vname, kw in variants.items():
        print(f"\n{'='*70}\n  {prefix} :: {vname}\n{'='*70}")
        torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
        model = cls(num_classes=NUM_CLASSES, **kw)
        tot, tr = count_parameters(model)
        model, _ = train_model(model, f"{prefix}_{vname}", lr_train_loader,
                               lr_val_loader, ABL_OUT_DIR, "ABL")
        m = evaluate_model(model, lr_test_loader, vname)
        m_pert = evaluate_model(model, lr_stain_loader, vname)["accuracy"]
        results[vname] = {"accuracy": m["accuracy"], "f1_macro": m["f1_macro"],
                          "kappa": m["cohen_kappa"], "mcc": m["mcc"],
                          "perturbed": m_pert,
                          "total_M": round(tot/1e6, 2), "trainable_M": round(tr/1e6, 3)}
        print(f"  acc={m['accuracy']:.4f}  perturbed={m_pert:.4f}  trainable={tr/1e6:.3f}M")
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return results


abl_dpct = run_ablation(DPCT_VARIANTS, DPCT_Ablation, "DPCT")
abl_msca = run_ablation(MSCA_VARIANTS, MSCA_Ablation, "MSCA")

abl_rows = []
for grp, res in [("DPCT-Net", abl_dpct), ("MSCA-Net", abl_msca)]:
    for v, r in res.items():
        abl_rows.append({"Architecture": grp, "Variant": v, **r})
abl_df = pd.DataFrame(abl_rows)
abl_df.to_csv(ABL_OUT_DIR / "ablation_results.csv", index=False)
print()
print(abl_df.to_string(index=False))

In [ ]:
# Ablation figure — clean and perturbed accuracy per variant
fig, axes = plt.subplots(1, 2, figsize=(17, 5.5))
for ax, (grp, res) in zip(axes, [("DPCT-Net", abl_dpct), ("MSCA-Net", abl_msca)]):
    vs = list(res.keys())
    x = np.arange(len(vs)); w = 0.38
    clean = [res[v]["accuracy"] for v in vs]
    pert  = [res[v]["perturbed"] for v in vs]
    colors = ["#C0392B" if "FULL" in v else "#4A90B8" for v in vs]
    ax.bar(x - w/2, clean, w, label="Clean", color=colors, alpha=0.9)
    ax.bar(x + w/2, pert,  w, label="Colour-perturbed", color=colors, alpha=0.5)
    for i, (c, p) in enumerate(zip(clean, pert)):
        ax.text(i - w/2, c + 0.004, f"{c:.4f}", ha="center", fontsize=7.5)
        ax.text(i + w/2, p + 0.004, f"{p:.4f}", ha="center", fontsize=7.5)
    ax.set_xticks(x); ax.set_xticklabels(vs, rotation=22, ha="right", fontsize=8.5)
    lo = min(min(clean), min(pert))
    ax.set_ylim(max(0, lo - 0.05), 1.01)
    ax.set_ylabel("Accuracy"); ax.set_title(f"{grp} component ablation")
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(ABL_OUT_DIR / "figures" / "ablation_results.png", dpi=300, bbox_inches="tight")
plt.show()

# contribution of each component relative to the full model
print("\nComponent contribution (full model minus variant, pp):")
for grp, res, full in [("DPCT-Net", abl_dpct, "A7_FULL_DPCT"),
                       ("MSCA-Net", abl_msca, "B5_FULL_MSCA")]:
    base = res[full]["accuracy"]
    print(f"\n  {grp}  (full = {base:.4f})")
    for v, r in res.items():
        if v == full: continue
        print(f"    {v:22s} {(base - r['accuracy'])*100:+6.2f} pp   "
              f"(perturbed {(res[full]['perturbed'] - r['perturbed'])*100:+6.2f} pp)")

---
# 18. Multi-seed evaluation with repeated splits

**R5-5** specifically asks for *inter-split* variance, not just training-seed
variance. So each seed regenerates the **group-aware split** as well as the
training initialisation — the group assignments stay fixed (they are data-derived)
while fold membership changes.

Six models × five seeds = 30 runs.

In [ ]:
MS_OUT_DIR = OUT_DIR / "multiseed"
(MS_OUT_DIR / "models").mkdir(parents=True, exist_ok=True)
(MS_OUT_DIR / "metrics").mkdir(parents=True, exist_ok=True)
(MS_OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

SEEDS = [42, 1, 7, 13, 2024]
MS_MODELS = ["EfficientNet_B3", "DenseNet121", "Swin_Tiny",
             "ConvNeXt_Tiny", "DPCT_Net", "MSCA_Net"]

print(f"Seeds : {SEEDS}")
print(f"Models: {MS_MODELS}")
print(f"Total runs: {len(SEEDS)*len(MS_MODELS)}")


def loaders_for_seed(seed):
    # regenerate the group-aware split for this seed
    tr, va, te = balanced_split(lc_df, "group_final", seed=seed)
    f = np.empty(len(lc_df), dtype=object)
    f[tr] = "train"; f[va] = "val"; f[te] = "test"
    d = lc_df.copy(); d["fold"] = f
    assert (d.groupby("group_final")["fold"].nunique() == 1).all()
    return (make_loader(SplitDataset(d, "train", CLASS_NAMES, train_transform), True),
            make_loader(SplitDataset(d, "val",   CLASS_NAMES, eval_transform),  False),
            make_loader(SplitDataset(d, "test",  CLASS_NAMES, eval_transform),  False))

In [ ]:
import gc, traceback

MS_CSV = MS_OUT_DIR / "multiseed_raw.csv"
ms_records = pd.read_csv(MS_CSV).to_dict("records") if MS_CSV.exists() else []
done = {(r["Model"], r["Seed"]) for r in ms_records}
print(f"Resuming — {len(done)} runs already recorded")

for seed in SEEDS:
    print(f"\n{'#'*70}\n#  SEED {seed}\n{'#'*70}")
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    tr_l, va_l, te_l = loaders_for_seed(seed)

    for name in MS_MODELS:
        if (name, seed) in done:
            print(f"--- {name} (seed {seed}) — already recorded, skipping")
            continue
        print(f"\n--- {name}  (seed {seed}) ---")
        model = None
        try:
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            model = MODELS[name](num_classes=NUM_CLASSES)
            model, _ = train_model(model, f"{name}_s{seed}", tr_l, va_l, MS_OUT_DIR, "MS")
            m = evaluate_model(model, te_l, name)
            ms_records.append({"Model": name, "Seed": seed,
                               "accuracy": m["accuracy"], "f1_macro": m["f1_macro"],
                               "kappa": m["cohen_kappa"], "mcc": m["mcc"]})
            pd.DataFrame(ms_records).to_csv(MS_CSV, index=False)   # save after each run
            print(f"  acc={m['accuracy']:.4f}  [saved]")
        except torch.cuda.OutOfMemoryError:
            print(f"  OOM on {name} (seed {seed}) — skipping, will retry on re-run")
        except Exception:
            traceback.print_exc()
        finally:
            if model is not None: del model
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

ms_df = pd.DataFrame(ms_records)
print(f"\nCollected {len(ms_df)}/{len(SEEDS)*len(MS_MODELS)} runs")
missing = [(m, s) for s in SEEDS for m in MS_MODELS if (m, s) not in
           {(r['Model'], r['Seed']) for r in ms_records}]
if missing: print(f"Still missing: {missing}")

In [ ]:
# aggregate: mean +/- std and 95% CI of the mean
from scipy import stats as sstats

agg = []
for name, g in ms_df.groupby("Model"):
    a = g["accuracy"].values
    n = len(a); mean = a.mean(); sd = a.std(ddof=1)
    ci = sstats.t.ppf(0.975, n-1) * sd / np.sqrt(n) if n > 1 else 0.0
    agg.append({"Model": name, "n_seeds": n,
                "Acc mean": round(mean, 4), "Acc std": round(sd, 4),
                "95% CI +/-": round(ci, 4),
                "Acc min": round(a.min(), 4), "Acc max": round(a.max(), 4),
                "Range (pp)": round((a.max()-a.min())*100, 2),
                "F1 mean": round(g["f1_macro"].mean(), 4),
                "F1 std": round(g["f1_macro"].std(ddof=1), 4)})
ms_agg = pd.DataFrame(agg).sort_values("Acc mean", ascending=False)
ms_agg.to_csv(MS_OUT_DIR / "multiseed_summary.csv", index=False)
print(ms_agg.to_string(index=False))

print("\nManuscript-ready (mean +/- std):")
for _, r in ms_agg.iterrows():
    print(f"  {r['Model']:18s} {r['Acc mean']:.4f} +/- {r['Acc std']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5.5))

order = list(ms_agg["Model"])
data = [ms_df[ms_df["Model"] == m]["accuracy"].values for m in order]
bp = axes[0].boxplot(data, labels=order, patch_artist=True, showmeans=True)
for p in bp['boxes']:
    p.set_facecolor("#4A90B8"); p.set_alpha(0.65)
for i, d in enumerate(data):
    axes[0].scatter([i+1]*len(d), d, color="#C0392B", s=28, zorder=3, alpha=0.8)
axes[0].set_ylabel("Test accuracy")
axes[0].set_title(f"Accuracy across {len(SEEDS)} seeds / repeated splits")
axes[0].tick_params(axis="x", rotation=22)
axes[0].grid(axis="y", alpha=0.3)

x = np.arange(len(ms_agg))
axes[1].errorbar(x, ms_agg["Acc mean"], yerr=ms_agg["95% CI +/-"], fmt="o",
                 capsize=6, capthick=1.8, markersize=9, color="#1C7293", ecolor="#C0392B")
for i, (_, r) in enumerate(ms_agg.iterrows()):
    axes[1].text(i, r["Acc mean"] + r["95% CI +/-"] + 0.0012,
                 f"{r['Acc mean']:.4f}", ha="center", fontsize=8.5)
axes[1].set_xticks(x); axes[1].set_xticklabels(ms_agg["Model"], rotation=22, ha="right")
axes[1].set_ylabel("Mean test accuracy")
axes[1].set_title("Mean with 95% confidence interval")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(MS_OUT_DIR / "figures" / "multiseed_variance.png", dpi=300, bbox_inches="tight")
plt.show()

# do any confidence intervals overlap?
print("\nOverlap check (are the top models distinguishable?):")
for i in range(len(ms_agg)):
    for j in range(i+1, len(ms_agg)):
        a, b = ms_agg.iloc[i], ms_agg.iloc[j]
        lo_a, hi_a = a["Acc mean"]-a["95% CI +/-"], a["Acc mean"]+a["95% CI +/-"]
        lo_b, hi_b = b["Acc mean"]-b["95% CI +/-"], b["Acc mean"]+b["95% CI +/-"]
        if not (hi_a < lo_b or hi_b < lo_a):
            print(f"  OVERLAP: {a['Model']} and {b['Model']} - not distinguishable")

---
# 19. Matched training regimes

Removes the frozen/fine-tuned confound. ResNet-50 and ConvNeXt-Tiny are each
trained under three regimes so architecture and optimisation are separable —
exactly what R2-5, R4-2 and R6-2 requested.

In [ ]:
REG_OUT_DIR = OUT_DIR / "regimes"
(REG_OUT_DIR / "models").mkdir(parents=True, exist_ok=True)
(REG_OUT_DIR / "metrics").mkdir(parents=True, exist_ok=True)
(REG_OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)


def make_resnet50(regime, num_classes=NUM_CLASSES):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    if regime == "linear_probe":
        for p in m.parameters(): p.requires_grad = False
    elif regime == "partial":
        for p in m.parameters(): p.requires_grad = False
        for p in m.layer4.parameters(): p.requires_grad = True
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m


def make_convnext(regime, num_classes=NUM_CLASSES):
    m = timm.create_model('convnext_tiny', pretrained=True, num_classes=num_classes)
    if regime == "linear_probe":
        for n_, p in m.named_parameters():
            p.requires_grad = n_.startswith("head")
    elif regime == "partial":
        for n_, p in m.named_parameters():
            p.requires_grad = n_.startswith("head") or "stages.3" in n_
    return m


REGIMES = ["linear_probe", "partial", "full"]
REGIME_BUILDERS = {"ResNet50": make_resnet50, "ConvNeXt_Tiny": make_convnext}

reg_rows = []
for arch, builder in REGIME_BUILDERS.items():
    for regime in REGIMES:
        tag = f"{arch}_{regime}"
        print(f"\n{'='*70}\n  {tag}\n{'='*70}")
        torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
        model = builder(regime)
        tot, tr = count_parameters(model)
        model, _ = train_model(model, tag, lr_train_loader, lr_val_loader,
                               REG_OUT_DIR, "REG")
        m = evaluate_model(model, lr_test_loader, tag)
        pert = evaluate_model(model, lr_stain_loader, tag)["accuracy"]
        reg_rows.append({"Architecture": arch, "Regime": regime,
                         "Total (M)": round(tot/1e6, 2),
                         "Trainable (M)": round(tr/1e6, 3),
                         "Trainable %": round(100*tr/tot, 2),
                         "Accuracy": round(m["accuracy"], 4),
                         "F1": round(m["f1_macro"], 4),
                         "Perturbed": round(pert, 4)})
        print(f"  acc={m['accuracy']:.4f}  perturbed={pert:.4f}  trainable={tr/1e6:.3f}M")
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

reg_df = pd.DataFrame(reg_rows)
reg_df.to_csv(REG_OUT_DIR / "matched_regimes.csv", index=False)
print()
print(reg_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
archs = reg_df["Architecture"].unique()
x = np.arange(len(REGIMES)); w = 0.36
for k, a in enumerate(archs):
    sub = reg_df[reg_df["Architecture"] == a].set_index("Regime").loc[REGIMES]
    b = ax.bar(x + (k - 0.5)*w, sub["Accuracy"], w, label=a, alpha=0.9)
    for xi, (acc, trn) in enumerate(zip(sub["Accuracy"], sub["Trainable (M)"])):
        ax.text(xi + (k - 0.5)*w, acc + 0.004, f"{acc:.4f}\n{trn:.2f}M",
                ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(["Linear probe", "Partial unfreeze", "Full fine-tune"])
ax.set_ylabel("Test accuracy")
ax.set_title("Matched training regimes — architecture vs optimisation")
ax.legend(); ax.grid(axis="y", alpha=0.3)
lo = reg_df["Accuracy"].min()
ax.set_ylim(max(0, lo - 0.06), 1.01)
plt.tight_layout()
plt.savefig(REG_OUT_DIR / "figures" / "matched_regimes.png", dpi=300, bbox_inches="tight")
plt.show()

print("Effect of regime within each architecture:")
for a in archs:
    sub = reg_df[reg_df["Architecture"] == a].set_index("Regime")
    span = (sub["Accuracy"].max() - sub["Accuracy"].min())*100
    print(f"  {a:16s} range across regimes = {span:.2f} pp")

---
# 20. DPCT-Net gating-weight distributions

**R6-6** asked that these be called *feature-fusion weights* rather than an
interpretability mechanism, and **R5-7** asked for class-wise distributions over
the full test set. This section provides both.

In [ ]:
GATE_OUT = OUT_DIR / "gating"
(GATE_OUT / "figures").mkdir(parents=True, exist_ok=True)

wpath = LR_OUT_DIR / "models" / "DPCT_Net_LR_best.pth"
assert wpath.exists(), f"Missing {wpath} — run Section 8 first"

gmodel = MODELS["DPCT_Net"](num_classes=NUM_CLASSES)
gmodel.load_state_dict(torch.load(wpath, map_location=device, weights_only=True))
gmodel.to(device).eval()

alphas, labels_g, preds_g = [], [], []
with torch.no_grad():
    for x, y in tqdm(lr_test_loader, desc="extracting gates"):
        out, g = gmodel(x.to(device), return_gate=True)
        alphas.append(g.cpu().numpy())
        labels_g.extend(y.numpy())
        preds_g.extend(out.argmax(1).cpu().numpy())

alphas = np.concatenate(alphas)
labels_g = np.array(labels_g); preds_g = np.array(preds_g)
gate_df = pd.DataFrame({"true_class": [CLASS_NAMES[i] for i in labels_g],
                        "pred_class": [CLASS_NAMES[i] for i in preds_g],
                        "alpha_cnn": alphas[:, 0], "alpha_transformer": alphas[:, 1],
                        "correct": labels_g == preds_g})
gate_df.to_csv(GATE_OUT / "dpct_gating_weights.csv", index=False)

print(f"Extracted gates for {len(gate_df)} test images\n")
print("Overall:")
print(f"  alpha_cnn         mean {alphas[:,0].mean():.4f}  std {alphas[:,0].std():.4f}")
print(f"  alpha_transformer mean {alphas[:,1].mean():.4f}  std {alphas[:,1].std():.4f}")
print("\nPer class:")
print(gate_df.groupby("true_class")[["alpha_cnn", "alpha_transformer"]]
      .agg(["mean", "std"]).round(4).to_string())
del gmodel
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# violin of alpha_cnn per class
data = [gate_df[gate_df["true_class"] == c]["alpha_cnn"].values for c in CLASS_NAMES]
vp = axes[0].violinplot(data, positions=range(len(CLASS_NAMES)), showmedians=True)
for b in vp['bodies']: b.set_facecolor("#4A90B8"); b.set_alpha(0.7)
axes[0].axhline(0.5, color="k", ls="--", lw=1, label="equal contribution")
axes[0].set_xticks(range(len(CLASS_NAMES)))
axes[0].set_xticklabels(CLASS_NAMES, rotation=25, ha="right", fontsize=9)
axes[0].set_ylabel(r"$\alpha_{CNN}$")
axes[0].set_title("CNN fusion weight by class")
axes[0].legend(fontsize=8); axes[0].grid(axis="y", alpha=0.3)

# histogram
axes[1].hist(alphas[:, 0], bins=50, color="#4A90B8", alpha=0.8)
axes[1].axvline(0.5, color="k", ls="--", lw=1)
axes[1].axvline(alphas[:, 0].mean(), color="#C0392B", lw=2,
                label=f"mean {alphas[:,0].mean():.3f}")
axes[1].set_xlabel(r"$\alpha_{CNN}$"); axes[1].set_ylabel("Count")
axes[1].set_title("Distribution over the test set")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

# correct vs incorrect
ok  = gate_df[gate_df["correct"]]["alpha_cnn"].values
bad = gate_df[~gate_df["correct"]]["alpha_cnn"].values
axes[2].hist(ok,  bins=40, alpha=0.65, density=True, label=f"correct (n={len(ok)})",
             color="#2E8B57")
if len(bad):
    axes[2].hist(bad, bins=40, alpha=0.65, density=True, label=f"incorrect (n={len(bad)})",
                 color="#C0392B")
axes[2].set_xlabel(r"$\alpha_{CNN}$"); axes[2].set_ylabel("Density")
axes[2].set_title("Fusion weight vs correctness")
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(GATE_OUT / "figures" / "dpct_gating_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

# is the class effect statistically real?
groups = [gate_df[gate_df["true_class"] == c]["alpha_cnn"].values for c in CLASS_NAMES]
F_stat, p_val = sstats.f_oneway(*groups)
print(f"One-way ANOVA across classes: F={F_stat:.3f}, p={p_val:.3e}")
print("  -> " + ("class-dependent fusion" if p_val < 0.05 else "no significant class effect"))
if len(bad):
    t_stat, p_t = sstats.ttest_ind(ok, bad, equal_var=False)
    print(f"Correct vs incorrect: t={t_stat:.3f}, p={p_t:.3e}")

---
# 21. Grad-CAM at scale with quantified agreement

**R6-7** objects that five images cannot demonstrate systematic behaviour, and
**R5-7** notes there is no quantitative validation. Without pathologist
annotations a pointing-game evaluation is impossible, so we report the strongest
available substitute: **inter-model agreement** between Grad-CAM maps, measured
as IoU of the top-20% activated regions across independently trained models.

High agreement means different architectures attend to the same tissue regions —
evidence of systematic behaviour rather than cherry-picked examples.

In [ ]:
CAM_OUT = OUT_DIR / "gradcam"
(CAM_OUT / "figures").mkdir(parents=True, exist_ok=True)

N_PER_CLASS = 20          # images sampled per class
CAM_MODELS = ["ConvNeXt_Tiny", "DenseNet121", "EfficientNet_B3", "MSCA_Net"]

def get_target_layer(model, name):
    try:
        if name == "VGG16":
            return [m for m in model.features if m.__class__.__name__ == "Conv2d"][-1]
        if name == "ResNet50":        return model.layer4[-1]
        if name == "DenseNet121":     return model.features.norm5
        if name == "MobileNetV3":     return model.conv_head
        if name in ("EfficientNet_B0", "EfficientNet_B3"): return model.conv_head
        if name == "ConvNeXt_Tiny":   return model.stages[-1]
        if name == "FSPAN_Y":         return model.y3
        if name == "MSCA_Net":        return model.proj[-1]
    except Exception:
        return None
    return None


class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        self.h = layer.register_forward_hook(self._fwd)
    def _fwd(self, m, i, o):
        self.acts = o
        if o.requires_grad:
            o.register_hook(lambda g: setattr(self, "grads", g.detach()))
    def __call__(self, x, idx):
        self.model.zero_grad()
        out = self.model(x)
        out[0, idx].backward()
        if self.grads is None: return None
        w = self.grads[0].mean(dim=(1, 2))
        cam = F.relu((w.view(-1, 1, 1) * self.acts[0].detach()).sum(0))
        cam -= cam.min()
        if cam.max() > 0: cam /= cam.max()
        return cam.cpu().numpy()
    def close(self): self.h.remove()


# sample a fixed set of test images
test_rows = lc_df[lc_df["fold"] == "test"]
rng = np.random.RandomState(SEED)
sample = pd.concat([g.iloc[rng.choice(len(g), min(N_PER_CLASS, len(g)), replace=False)]
                    for _, g in test_rows.groupby("class")])
print(f"Sampled {len(sample)} test images ({N_PER_CLASS} per class)")

In [ ]:
# compute CAMs for every (model, image)
cams = {m: {} for m in CAM_MODELS}
for name in CAM_MODELS:
    w = LR_OUT_DIR / "models" / f"{name}_LR_best.pth"
    if not w.exists():
        print(f"  skip {name}: weights missing"); continue
    model = MODELS[name](num_classes=NUM_CLASSES)
    model.load_state_dict(torch.load(w, map_location=device, weights_only=True))
    model.to(device).eval()
    layer = get_target_layer(model, name)
    if layer is None:
        print(f"  skip {name}: no target layer"); del model; continue

    cam_extractor = GradCAM(model, layer)          # renamed from gc
    for ridx, (_, r) in enumerate(tqdm(sample.iterrows(), total=len(sample),
                                       desc=f"Grad-CAM {name}", leave=False)):
        img = Image.open(r["path"]).convert("RGB")
        x = eval_transform(img).unsqueeze(0).to(device).requires_grad_(True)
        cam = cam_extractor(x, CLASS_NAMES.index(r["class"]))
        if cam is not None:
            cams[name][ridx] = cam
    cam_extractor.close(); del model, cam_extractor
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"  {name}: {len(cams[name])} CAMs")

In [ ]:
# inter-model agreement: IoU of top-20% activated regions
from itertools import combinations

def topk_mask(cam, frac=0.20):
    c = cam.astype(np.float32)
    thr = np.quantile(c, 1 - frac)
    return c >= thr

def resize_to(a, shape):
    im = Image.fromarray((a * 255).astype(np.uint8)).resize((shape[1], shape[0]),
                                                            Image.BILINEAR)
    return np.array(im) / 255.0

avail = [m for m in CAM_MODELS if len(cams[m]) > 0]
iou_rows = []
for m1, m2 in combinations(avail, 2):
    shared = set(cams[m1]) & set(cams[m2])
    ious = []
    for i in shared:
        a, b = cams[m1][i], cams[m2][i]
        if a.shape != b.shape:
            b = resize_to(b, a.shape)
        ma, mb = topk_mask(a), topk_mask(b)
        inter = np.logical_and(ma, mb).sum(); union = np.logical_or(ma, mb).sum()
        if union > 0: ious.append(inter / union)
    if ious:
        iou_rows.append({"Model A": m1, "Model B": m2, "n_images": len(ious),
                         "mean IoU": round(float(np.mean(ious)), 4),
                         "std IoU": round(float(np.std(ious)), 4),
                         "median IoU": round(float(np.median(ious)), 4)})

iou_df = pd.DataFrame(iou_rows).sort_values("mean IoU", ascending=False)
iou_df.to_csv(CAM_OUT / "gradcam_agreement.csv", index=False)
print(iou_df.to_string(index=False))
if len(iou_df):
    print(f"\nMean inter-model IoU across all pairs: {iou_df['mean IoU'].mean():.4f}")
    print("Random-chance IoU for two independent top-20% masks ~ 0.111")

In [ ]:
# qualitative grid: one row per class, one column per model
def denorm(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1); std = torch.tensor(IMAGENET_STD).view(3,1,1)
    return ((t.cpu()*std + mean).clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)

picks = {}
for ci, cls in enumerate(CLASS_NAMES):
    idxs = [i for i, (_, r) in enumerate(sample.iterrows()) if r["class"] == cls]
    if idxs: picks[cls] = idxs[0]

ncol = 1 + len(avail)
fig, axes = plt.subplots(len(picks), ncol, figsize=(3.1*ncol, 3.1*len(picks)))
axes = np.atleast_2d(axes)
for ri, (cls, si) in enumerate(picks.items()):
    row = sample.iloc[si]
    img = Image.open(row["path"]).convert("RGB")
    x = eval_transform(img)
    axes[ri, 0].imshow(denorm(x)); axes[ri, 0].set_ylabel(cls, fontsize=9)
    axes[ri, 0].set_xticks([]); axes[ri, 0].set_yticks([])
    if ri == 0: axes[ri, 0].set_title("Original", fontsize=10)
    for ci, name in enumerate(avail, start=1):
        ax = axes[ri, ci]; ax.set_xticks([]); ax.set_yticks([])
        if si in cams[name]:
            cam = resize_to(cams[name][si], (IMG_SIZE, IMG_SIZE))
            ax.imshow(denorm(x)); ax.imshow(cam, cmap="jet", alpha=0.45)
        else:
            ax.axis("off")
        if ri == 0: ax.set_title(name, fontsize=10)
plt.suptitle("Grad-CAM across independently trained models", y=1.002, fontsize=13)
plt.tight_layout()
plt.savefig(CAM_OUT / "figures" / "gradcam_grid.png", dpi=300, bbox_inches="tight")
plt.show()

---
# 22. Part II summary

In [ ]:
print("="*72)
print("PART II — REVIEWER RESPONSE SUMMARY")
print("="*72)

if "abl_df" in dir():
    print("\n[17] ABLATIONS")
    for grp, full in [("DPCT-Net", "A7_FULL_DPCT"), ("MSCA-Net", "B5_FULL_MSCA")]:
        sub = abl_df[abl_df["Architecture"] == grp]
        if len(sub) == 0: continue
        best = sub.loc[sub["accuracy"].idxmax()]
        fullrow = sub[sub["Variant"] == full]
        print(f"  {grp}: best variant = {best['Variant']} ({best['accuracy']:.4f})")
        if len(fullrow):
            fr = fullrow.iloc[0]
            print(f"    full model = {fr['accuracy']:.4f}, "
                  f"{'CONFIRMS' if fr['Variant']==best['Variant'] else 'DOES NOT confirm'} "
                  f"the complete design")

if "ms_agg" in dir():
    print("\n[18] MULTI-SEED")
    print(f"  Mean std across models: {ms_agg['Acc std'].mean():.4f}")
    print(f"  Largest seed range    : {ms_agg['Range (pp)'].max():.2f} pp")
    top = ms_agg.iloc[0]
    print(f"  Best: {top['Model']} {top['Acc mean']:.4f} +/- {top['Acc std']:.4f}")

if "reg_df" in dir():
    print("\n[19] MATCHED REGIMES")
    for a in reg_df["Architecture"].unique():
        s = reg_df[reg_df["Architecture"] == a]
        print(f"  {a}: {s['Accuracy'].min():.4f} -> {s['Accuracy'].max():.4f} "
              f"({(s['Accuracy'].max()-s['Accuracy'].min())*100:.2f} pp across regimes)")

if "gate_df" in dir():
    print("\n[20] GATING")
    print(f"  alpha_cnn mean {gate_df['alpha_cnn'].mean():.4f} "
          f"std {gate_df['alpha_cnn'].std():.4f}")

if "iou_df" in dir() and len(iou_df):
    print("\n[21] GRAD-CAM AGREEMENT")
    print(f"  Mean inter-model IoU: {iou_df['mean IoU'].mean():.4f} (chance ~0.111)")

print("="*72)

---
# 21B. DPCT-Net Grad-CAM (spatial-faithful variant)

DPCT-Net as trained cannot produce Grad-CAM: both backbones are built with
`global_pool='avg'` (output is a 768-d vector, no spatial map) and run inside
`torch.no_grad()` (no gradient path).

`DPCT_Net_CAM` below solves this **without changing the model**:

* identical submodule names, so it loads the *exact same* `state_dict`;
* calls `forward_features()` to retain the spatial map, then
  `forward_head(pre_logits=True)` to pool it — mathematically identical to the
  original `self.cnn(x)` / `self.tfm(x)`;
* drops the `no_grad` wrapper so gradients reach the backbone activations.
  Frozen parameters stay frozen — `requires_grad=False` on weights does not
  block gradient flow through activations when the input requires grad.

The first cell **proves numerical equivalence** before any CAM is computed. If
the assertion fails, the CAMs are not faithful and must not be used.

In [ ]:
class DPCT_Net_CAM(nn.Module):
    # CAM-instrumented DPCT-Net. Same parameters, same maths, spatial maps kept.
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.cnn = timm.create_model('convnext_tiny', pretrained=False,
                                     num_classes=0, global_pool='avg')
        self.tfm = timm.create_model('swin_tiny_patch4_window7_224', pretrained=False,
                                     num_classes=0, global_pool='avg')
        for p in self.cnn.parameters(): p.requires_grad = False
        for p in self.tfm.parameters(): p.requires_grad = False
        d = 256
        self.proj_cnn = nn.Sequential(nn.Linear(self.cnn.num_features, d),
                                      nn.GELU(), nn.LayerNorm(d))
        self.proj_tfm = nn.Sequential(nn.Linear(self.tfm.num_features, d),
                                      nn.GELU(), nn.LayerNorm(d))
        self.cross_attn = nn.MultiheadAttention(d, num_heads=4, batch_first=True)
        self.gate = nn.Sequential(nn.Linear(d*2, d), nn.GELU(),
                                  nn.Linear(d, 2), nn.Softmax(dim=-1))
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(d, num_classes))
        # populated on every forward pass
        self.cnn_map = None
        self.tfm_map = None

    def forward(self, x, return_gate=False):
        # ---- CNN stream: keep the spatial map ----
        fmap_c = self.cnn.forward_features(x)          # (B, C, H, W)
        self.cnn_map = fmap_c
        f_cnn = self.cnn.forward_head(fmap_c, pre_logits=True)

        # ---- Transformer stream: keep the spatial map ----
        fmap_t = self.tfm.forward_features(x)          # (B,H,W,C) or (B,C,H,W)
        self.tfm_map = fmap_t
        f_tfm = self.tfm.forward_head(fmap_t, pre_logits=True)

        # ---- identical to the original from here on ----
        z_cnn = self.proj_cnn(f_cnn)
        z_tfm = self.proj_tfm(f_tfm)
        seq = torch.stack([z_cnn, z_tfm], dim=1)
        attn_out, _ = self.cross_attn(seq, seq, seq)
        g = self.gate(torch.cat([z_cnn, z_tfm], dim=-1))
        fused = g[:, 0:1] * attn_out[:, 0] + g[:, 1:2] * attn_out[:, 1]
        out = self.classifier(fused)
        return (out, g) if return_gate else out


# ══════════════════════════════════════════════════════════════════
# EQUIVALENCE PROOF — the CAM variant must reproduce the trained model
# ══════════════════════════════════════════════════════════════════
w = LR_OUT_DIR / "models" / "DPCT_Net_LR_best.pth"
assert w.exists(), f"missing {w}"
sd = torch.load(w, map_location="cpu", weights_only=True)

m_orig = MODELS["DPCT_Net"](num_classes=NUM_CLASSES)
m_cam  = DPCT_Net_CAM(num_classes=NUM_CLASSES)

r1 = m_orig.load_state_dict(sd, strict=True)
r2 = m_cam.load_state_dict(sd, strict=True)
print("state_dict loaded into both models with strict=True")

m_orig.to(device).eval(); m_cam.to(device).eval()

# same input, deterministic (eval mode disables dropout)
torch.manual_seed(0)
xb, _ = next(iter(lr_test_loader))
xb = xb[:8].to(device)

with torch.no_grad():
    o1, g1 = m_orig(xb, return_gate=True)
    o2, g2 = m_cam(xb,  return_gate=True)

max_logit_diff = (o1 - o2).abs().max().item()
max_gate_diff  = (g1 - g2).abs().max().item()
print(f"\nmax |logit difference| : {max_logit_diff:.3e}")
print(f"max |gate  difference| : {max_gate_diff:.3e}")
print(f"predictions identical  : {bool((o1.argmax(1) == o2.argmax(1)).all())}")
print(f"CNN spatial map shape  : {tuple(m_cam.cnn_map.shape)}")
print(f"TFM spatial map shape  : {tuple(m_cam.tfm_map.shape)}")

assert max_logit_diff < 1e-4, "CAM variant is NOT equivalent — do not use its CAMs"
print("\nEQUIVALENCE VERIFIED — CAMs from DPCT_Net_CAM are faithful to the trained model")

del m_orig
if torch.cuda.is_available(): torch.cuda.empty_cache()

### 21B.2 Grad-CAM for both DPCT-Net streams

Because DPCT-Net fuses two pathways, we compute a separate CAM for each and
report them alongside the gate weight for that image. This is more informative
than a single map and directly supports the gating analysis in Section 20.

In [ ]:
def _as_bchw(t):
    # Normalise a feature map to (B, C, H, W).
    # ConvNeXt gives (B,C,H,W); Swin gives (B,H,W,C) or sometimes (B,L,C).
    if t is None:
        return None
    if t.dim() == 4:
        b, a1, a2, a3 = t.shape
        if a1 == a2 and a3 != a1:          # (B,H,W,C) -> (B,C,H,W)
            return t.permute(0, 3, 1, 2)
        return t                            # already (B,C,H,W)
    if t.dim() == 3:                        # (B,L,C) -> (B,C,sqrt(L),sqrt(L))
        b, L, C = t.shape
        s = int(round(L ** 0.5))
        if s * s == L:
            return t.transpose(1, 2).reshape(b, C, s, s)
    return None


def dpct_cams(model, x, class_idx):
    # Returns (cam_cnn, cam_tfm, gate) for a single image tensor x (1,3,H,W).
    model.zero_grad(set_to_none=True)
    x = x.clone().requires_grad_(True)
    out, gate = model(x, return_gate=True)

    # retain_grad MUST be called on the tensors that are actually in the graph,
    # not on reshaped copies made afterwards.
    raw_c, raw_t = model.cnn_map, model.tfm_map
    raw_c.retain_grad()
    raw_t.retain_grad()

    out[0, class_idx].backward()

    cams = []
    for raw in (raw_c, raw_t):
        if raw.grad is None:
            cams.append(None); continue
        act  = _as_bchw(raw.detach())
        grad = _as_bchw(raw.grad)
        if act is None or grad is None:
            cams.append(None); continue
        wts = grad[0].mean(dim=(1, 2))                       # (C,)
        cam = F.relu((wts.view(-1, 1, 1) * act[0]).sum(0))   # (H,W)
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        cams.append(cam.cpu().numpy())
    return cams[0], cams[1], gate[0].detach().cpu().numpy()


# sanity check on one image
_row = sample.iloc[0]
_x = eval_transform(Image.open(_row["path"]).convert("RGB")).unsqueeze(0).to(device)
_cc, _ct, _g = dpct_cams(m_cam, _x, CLASS_NAMES.index(_row["class"]))
print(f"raw cnn_map shape : {tuple(m_cam.cnn_map.shape)}")
print(f"raw tfm_map shape : {tuple(m_cam.tfm_map.shape)}")
print(f"CNN CAM shape {None if _cc is None else _cc.shape} | "
      f"TFM CAM shape {None if _ct is None else _ct.shape}")
print(f"gate (alpha_cnn, alpha_tfm) = {np.round(_g, 4)}")
assert _cc is not None and _ct is not None, "gradients did not reach a backbone map"
print("Both DPCT-Net streams produce valid Grad-CAMs")

In [ ]:
# compute DPCT-Net CAMs over the same 100-image sample used in Section 21
cams["DPCT_Net_CNN"] = {}
cams["DPCT_Net_TFM"] = {}
dpct_gates = []

for ridx, (_, r) in enumerate(tqdm(sample.iterrows(), total=len(sample),
                                   desc="Grad-CAM DPCT_Net", leave=False)):
    img = Image.open(r["path"]).convert("RGB")
    x = eval_transform(img).unsqueeze(0).to(device)
    cc, ct, g = dpct_cams(m_cam, x, CLASS_NAMES.index(r["class"]))
    if cc is not None: cams["DPCT_Net_CNN"][ridx] = cc
    if ct is not None: cams["DPCT_Net_TFM"][ridx] = ct
    dpct_gates.append({"idx": ridx, "class": r["class"],
                       "alpha_cnn": float(g[0]), "alpha_tfm": float(g[1])})

dpct_gate_df = pd.DataFrame(dpct_gates)
dpct_gate_df.to_csv(CAM_OUT / "dpct_cam_gates.csv", index=False)
print(f"DPCT_Net_CNN: {len(cams['DPCT_Net_CNN'])} CAMs")
print(f"DPCT_Net_TFM: {len(cams['DPCT_Net_TFM'])} CAMs")
print("\nGate weights on the CAM sample, per class:")
print(dpct_gate_df.groupby("class")[["alpha_cnn", "alpha_tfm"]].mean().round(4).to_string())

### 21B.3 Updated inter-model agreement including DPCT-Net

In [ ]:
from itertools import combinations

CAM_ALL = [m for m in list(cams.keys()) if len(cams[m]) > 0]
print(f"Models with CAMs: {CAM_ALL}\n")

iou_rows2 = []
for m1, m2 in combinations(CAM_ALL, 2):
    shared = set(cams[m1]) & set(cams[m2])
    ious = []
    for i in shared:
        a, b = cams[m1][i], cams[m2][i]
        if a.shape != b.shape:
            b = resize_to(b, a.shape)
        ma, mb = topk_mask(a), topk_mask(b)
        u = np.logical_or(ma, mb).sum()
        if u > 0: ious.append(np.logical_and(ma, mb).sum() / u)
    if ious:
        iou_rows2.append({"Model A": m1, "Model B": m2, "n_images": len(ious),
                          "mean IoU": round(float(np.mean(ious)), 4),
                          "std IoU": round(float(np.std(ious)), 4),
                          "median IoU": round(float(np.median(ious)), 4)})

iou_all = pd.DataFrame(iou_rows2).sort_values("mean IoU", ascending=False)
iou_all.to_csv(CAM_OUT / "gradcam_agreement_with_dpct.csv", index=False)
print(iou_all.to_string(index=False))
print(f"\nMean IoU over all pairs: {iou_all['mean IoU'].mean():.4f}   (chance ~0.111)")

# how do the two DPCT streams compare with each other?
sub = iou_all[((iou_all['Model A'] == 'DPCT_Net_CNN') & (iou_all['Model B'] == 'DPCT_Net_TFM')) |
              ((iou_all['Model A'] == 'DPCT_Net_TFM') & (iou_all['Model B'] == 'DPCT_Net_CNN'))]
if len(sub):
    print(f"\nDPCT-Net CNN vs Transformer stream IoU: {sub.iloc[0]['mean IoU']:.4f}")
    print("  (low value = the two streams attend to different regions)")

In [ ]:
# ── Figure: DPCT-Net Grad-CAM, both streams, one row per class ──
picks2 = {}
for cls in CLASS_NAMES:
    idxs = [i for i, (_, r) in enumerate(sample.iterrows()) if r["class"] == cls]
    if idxs: picks2[cls] = idxs[0]

fig, axes = plt.subplots(len(picks2), 3, figsize=(10.5, 3.3*len(picks2)))
axes = np.atleast_2d(axes)
for ri, (cls, si) in enumerate(picks2.items()):
    row = sample.iloc[si]
    x = eval_transform(Image.open(row["path"]).convert("RGB"))
    base = denorm(x)
    g = dpct_gate_df[dpct_gate_df["idx"] == si].iloc[0]

    axes[ri, 0].imshow(base); axes[ri, 0].set_ylabel(cls, fontsize=9)
    axes[ri, 0].set_xticks([]); axes[ri, 0].set_yticks([])
    if ri == 0: axes[ri, 0].set_title("Original", fontsize=11)

    for ci, (key, lab) in enumerate([("DPCT_Net_CNN", "CNN stream"),
                                     ("DPCT_Net_TFM", "Transformer stream")], start=1):
        ax = axes[ri, ci]; ax.set_xticks([]); ax.set_yticks([])
        if si in cams[key]:
            cam = resize_to(cams[key][si], (IMG_SIZE, IMG_SIZE))
            ax.imshow(base); ax.imshow(cam, cmap="jet", alpha=0.45)
            a = g["alpha_cnn"] if key.endswith("CNN") else g["alpha_tfm"]
            ax.set_xlabel(f"$\\alpha$ = {a:.3f}", fontsize=9)
        else:
            ax.axis("off")
        if ri == 0: ax.set_title(lab, fontsize=11)

plt.suptitle("DPCT-Net Grad-CAM — both fusion streams with gate weights",
             y=1.002, fontsize=13)
plt.tight_layout()
plt.savefig(CAM_OUT / "figures" / "dpct_gradcam_streams.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved dpct_gradcam_streams.png  — this replaces Figure 12's DPCT-Net panel")

In [ ]:
# ── Combined figure: all models side by side (Figure 12 replacement) ──
GRID = [m for m in ["ConvNeXt_Tiny", "DenseNet121", "EfficientNet_B3",
                    "MSCA_Net", "DPCT_Net_CNN", "DPCT_Net_TFM"] if m in CAM_ALL]
ncol = 1 + len(GRID)
fig, axes = plt.subplots(len(picks2), ncol, figsize=(2.9*ncol, 3.0*len(picks2)))
axes = np.atleast_2d(axes)
for ri, (cls, si) in enumerate(picks2.items()):
    row = sample.iloc[si]
    x = eval_transform(Image.open(row["path"]).convert("RGB"))
    base = denorm(x)
    axes[ri, 0].imshow(base); axes[ri, 0].set_ylabel(cls, fontsize=9)
    axes[ri, 0].set_xticks([]); axes[ri, 0].set_yticks([])
    if ri == 0: axes[ri, 0].set_title("Original", fontsize=10)
    for ci, name in enumerate(GRID, start=1):
        ax = axes[ri, ci]; ax.set_xticks([]); ax.set_yticks([])
        if si in cams[name]:
            cam = resize_to(cams[name][si], (IMG_SIZE, IMG_SIZE))
            ax.imshow(base); ax.imshow(cam, cmap="jet", alpha=0.45)
        else:
            ax.axis("off")
        if ri == 0: ax.set_title(name.replace("_", " "), fontsize=10)
plt.suptitle("Grad-CAM across models (n = 20 images per class analysed; one shown)",
             y=1.002, fontsize=13)
plt.tight_layout()
plt.savefig(CAM_OUT / "figures" / "gradcam_grid_full.png", dpi=300, bbox_inches="tight")
plt.show()

del m_cam
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("Saved gradcam_grid_full.png")

### 21B.4 What to write in Section 4.8

The descriptive claims in the current manuscript were not produced by a
spatially faithful DPCT-Net Grad-CAM and must be rewritten from the figures
generated above. Report:

1. that DPCT-Net fuses after global pooling, so CAMs are computed per stream
   on the pre-pooling feature maps, with numerical equivalence to the deployed
   model verified;
2. the gate weight for each displayed image, linking Figure 12 to Section 20;
3. the inter-model IoU as the quantitative support, with its chance baseline;
4. that 20 images per class were analysed and one representative image per
   class is displayed — not that conclusions rest on five images.

Avoid asserting specific anatomical correspondence unless a pathologist has
reviewed the maps, as R5-7 and R6-7 requested.

---
---
# PART III — Gap-Closing Experiments

Three remaining reviewer requirements.

| Section | Closes | Raised by |
|---|---|---|
| 23 | FLOPs, peak VRAM, throughput, checkpoint size | R5-6, R6-10 |
| 24 | Colour-jitter intensity calibration | R2-3 |
| 25 | Proposed models under matched training regimes | R2-5 |

Runtime: Section 23 ≈ 15 min, Section 24 ≈ 40 min, Section 25 ≈ 2 h.

---
# 23. Computational cost: FLOPs, memory, throughput

R5-6 and R6-10 both object that trainable-parameter count is not deployment
cost. This section measures what actually matters for deployment:

* **GFLOPs** per forward pass (and GMACs, since conventions differ)
* **Peak GPU memory** at batch 1 and batch 32
* **Throughput** in images/second
* **Checkpoint size** on disk

FLOPs are measured with PyTorch's built-in `FlopCounterMode` (no extra
dependency). Note that frozen backbones still incur full inference cost —
this is the point the reviewers are making about DPCT-Net.

In [ ]:
COST_OUT = OUT_DIR / "cost"
COST_OUT.mkdir(parents=True, exist_ok=True)

try:
    from torch.utils.flop_counter import FlopCounterMode
    _HAS_FLOP = True
except Exception:
    _HAS_FLOP = False
    print("FlopCounterMode unavailable — FLOPs will be reported as NaN")


@torch.no_grad()
def measure_cost(model, name, batch_sizes=(1, 32), n_warmup=10, n_timed=40):
    model = model.to(device).eval()
    out = {"Model": name}

    # ---- FLOPs (batch 1) ----
    out["GFLOPs"] = float("nan")
    if _HAS_FLOP:
        try:
            x1 = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
            fc = FlopCounterMode(display=False)
            with fc:
                model(x1)
            total = fc.get_total_flops()
            out["GFLOPs"] = round(total / 1e9, 3)
            out["GMACs"]  = round(total / 2e9, 3)
        except Exception as e:
            print(f"    FLOP count failed for {name}: {type(e).__name__}")

    # ---- peak memory + throughput ----
    for bs in batch_sizes:
        try:
            x = torch.randn(bs, 3, IMG_SIZE, IMG_SIZE, device=device)
            if torch.cuda.is_available():
                torch.cuda.synchronize(); torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
            for _ in range(n_warmup):
                model(x)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(n_timed):
                model(x)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            dt = (time.time() - t0) / n_timed
            out[f"ms/img @bs{bs}"] = round(1000 * dt / bs, 3)
            out[f"img/s @bs{bs}"]  = round(bs / dt, 1)
            if torch.cuda.is_available():
                out[f"peak VRAM MB @bs{bs}"] = round(
                    torch.cuda.max_memory_allocated() / 1024**2, 1)
        except torch.cuda.OutOfMemoryError:
            out[f"ms/img @bs{bs}"] = float("nan")
            out[f"img/s @bs{bs}"] = float("nan")
            out[f"peak VRAM MB @bs{bs}"] = float("nan")
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            print(f"    OOM at batch {bs} for {name}")
    return out


cost_rows = []
for name, fn in MODELS.items():
    print(f"  measuring {name} ...")
    m = fn(num_classes=NUM_CLASSES)
    row = measure_cost(m, name)

    # checkpoint size on disk
    ck = LR_OUT_DIR / "models" / f"{name}_LR_best.pth"
    row["Checkpoint MB"] = round(ck.stat().st_size / 1024**2, 1) if ck.exists() else float("nan")

    tot, tr = count_parameters(m)
    row["Total (M)"] = round(tot/1e6, 2)
    row["Trainable (M)"] = round(tr/1e6, 3)
    cost_rows.append(row)
    del m
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

cost_df = pd.DataFrame(cost_rows)
cost_df.to_csv(COST_OUT / "computational_cost.csv", index=False)
print()
print(cost_df.to_string(index=False))

In [ ]:
# merge cost with accuracy for the deployment table
dep = cost_df.copy()
dep["Family"]    = dep["Model"].map(MODEL_FAMILY)
dep["Acc (LR)"]  = dep["Model"].map(lambda n: round(lr_metrics[n]["accuracy"], 4)
                                    if n in lr_metrics else np.nan)
if "stain_df" in dir() and len(stain_df):
    _sm = dict(zip(stain_df["Model"], stain_df["Perturbed"]))
    dep["Perturbed"] = dep["Model"].map(_sm)

cols = ["Model", "Family", "Total (M)", "Trainable (M)", "GFLOPs",
        "peak VRAM MB @bs32", "img/s @bs32", "Checkpoint MB", "Acc (LR)"]
if "Perturbed" in dep.columns: cols.append("Perturbed")
cols = [c for c in cols if c in dep.columns]
dep_tbl = dep[cols].sort_values("GFLOPs")
dep_tbl.to_csv(COST_OUT / "deployment_table.csv", index=False)
print(dep_tbl.to_string(index=False))

print("\nKey comparisons for the manuscript:")
for a, b in [("DPCT_Net", "MobileNetV3"), ("MSCA_Net", "ConvNeXt_Tiny")]:
    if a in dep["Model"].values and b in dep["Model"].values:
        ra = dep[dep["Model"] == a].iloc[0]; rb = dep[dep["Model"] == b].iloc[0]
        if not np.isnan(ra["GFLOPs"]) and not np.isnan(rb["GFLOPs"]):
            print(f"  {a} needs {ra['GFLOPs']/rb['GFLOPs']:.1f}x the FLOPs of {b} "
                  f"({ra['GFLOPs']:.2f} vs {rb['GFLOPs']:.2f} GFLOPs) "
                  f"despite {ra['Trainable (M)']:.3f}M vs {rb['Trainable (M)']:.3f}M trainable")

In [ ]:
from adjustText import adjust_text

# Figure: trainable params vs true compute cost
fig, axes = plt.subplots(1, 2, figsize=(16, 5.6))
fam_colors = {"Classic CNN":"#E07B54","Efficient CNN":"#F5C242","Modern CNN":"#4A90B8",
              "Transformer":"#7B68EE","Hybrid (existing)":"#3DBD91","Hybrid (proposed)":"#C0392B"}
sub = dep.dropna(subset=["GFLOPs"])

for ax, (xcol, xlab) in zip(axes, [("Trainable (M)", "Trainable parameters (M)"),
                                   ("GFLOPs", "GFLOPs per image")]):
    
    texts = [] # Create an empty list for the current subplot's text labels
    
    for _, r in sub.iterrows():
        # Draw the scatter point
        ax.scatter(r[xcol], r["Acc (LR)"], s=90,
                   color=fam_colors.get(r["Family"], "#888"),
                   edgecolors="white", linewidths=1.2, zorder=3)
        
        # Add the text directly on the point and save it to the list
        t = ax.text(r[xcol], r["Acc (LR)"], r["Model"], fontsize=7.5, zorder=4)
        texts.append(t)
        
    ax.set_xscale("log")
    ax.set_xlabel(xlab)
    ax.set_ylabel("Accuracy (leakage-resistant)")
    ax.grid(alpha=0.3)
    
    # Run the text adjustment algorithm for this specific subplot
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-|>", color="gray", lw=0.6, alpha=0.7))

axes[0].set_title("Trainable parameters — the misleading view")
axes[1].set_title("FLOPs — the deployment-relevant view")

handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=c,
                      markersize=9, label=f) for f, c in fam_colors.items()]
axes[1].legend(handles=handles, fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig(COST_OUT / "trainable_vs_flops.png", dpi=300, bbox_inches="tight")
plt.show()
print("This figure is the direct answer to R5-6 and R6-10.")

---
# 24. Colour-jitter intensity calibration

R2-3 asks us to justify the perturbation strength. Rather than assert a value,
we sweep it and test whether the **model ranking is stable** across intensities.
A ranking invariant to intensity means the conclusion does not depend on an
arbitrary choice — the same logic used for the merge threshold in Section 3.

All models see **identical perturbations**: the RNG is re-seeded before each
evaluation pass, and `NUM_WORKERS=0` makes the transform draw deterministic.

In [ ]:
JIT_OUT = OUT_DIR / "jitter_sweep"
JIT_OUT.mkdir(parents=True, exist_ok=True)

JITTER_LEVELS = [0.00, 0.10, 0.20, 0.35, 0.50]
HUE_RATIO = JITTER_HUE / JITTER_BCS          # preserves the original 0.08/0.35 ratio


def make_jitter_loader(strength):
    if strength <= 0:
        tf = eval_transform
    else:
        tf = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ColorJitter(brightness=strength, contrast=strength,
                                   saturation=strength, hue=strength*HUE_RATIO),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return make_loader(SplitDataset(lc_df, "test", CLASS_NAMES, tf), False)


JIT_CSV = JIT_OUT / "jitter_sweep_raw.csv"
jit_records = pd.read_csv(JIT_CSV).to_dict("records") if JIT_CSV.exists() else []
done_j = {(r["Model"], r["strength"]) for r in jit_records}
print(f"Resuming — {len(done_j)} evaluations already recorded")

for s in JITTER_LEVELS:
    loader = make_jitter_loader(s)
    print(f"\n--- jitter strength {s:.2f} (hue {s*HUE_RATIO:.3f}) ---")
    for name, fn in MODELS.items():
        if (name, s) in done_j:
            continue
        w = LR_OUT_DIR / "models" / f"{name}_LR_best.pth"
        if not w.exists():
            print(f"  skip {name}: no weights"); continue
        model = fn(num_classes=NUM_CLASSES)
        model.load_state_dict(torch.load(w, map_location=device, weights_only=True))
        model.to(device).eval()

        # identical perturbation draw for every model
        torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
        acc = evaluate_model(model, loader, name)["accuracy"]

        jit_records.append({"Model": name, "strength": s, "accuracy": acc})
        pd.DataFrame(jit_records).to_csv(JIT_CSV, index=False)
        print(f"  {name:18s} {acc:.4f}")
        del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

jit_df = pd.DataFrame(jit_records)
print(f"\nCollected {len(jit_df)} evaluations")

In [ ]:
# pivot: models x intensities
pivot = jit_df.pivot(index="Model", columns="strength", values="accuracy")
pivot = pivot.sort_values(JITTER_LEVELS[-1], ascending=False).round(4)
pivot.to_csv(JIT_OUT / "jitter_sweep_matrix.csv")
print("Accuracy by jitter strength:")
print(pivot.to_string())

# rank stability across intensities
from scipy.stats import spearmanr
ranks = pivot.rank(ascending=False)
print("\nSpearman rank correlation between intensities:")
base = JITTER_LEVELS[-2] if len(JITTER_LEVELS) > 1 else JITTER_LEVELS[0]
rho_rows = []
for s in JITTER_LEVELS:
    for s2 in JITTER_LEVELS:
        if s >= s2: continue
        rho, p = spearmanr(pivot[s], pivot[s2])
        rho_rows.append({"strength A": s, "strength B": s2,
                         "spearman rho": round(float(rho), 4),
                         "p": round(float(p), 6)})
rho_df = pd.DataFrame(rho_rows)
rho_df.to_csv(JIT_OUT / "rank_stability.csv", index=False)
print(rho_df.to_string(index=False))
print(f"\nMean rho across all intensity pairs: {rho_df['spearman rho'].mean():.4f}")
print("rho near 1.0 means the model ranking is invariant to the chosen intensity.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5.6))

for name in pivot.index:
    vals = [pivot.loc[name, s] for s in JITTER_LEVELS]
    lw = 2.6 if name in ("DPCT_Net", "MSCA_Net") else 1.3
    axes[0].plot(JITTER_LEVELS, vals, marker="o", lw=lw, label=name, alpha=0.9)
axes[0].axvline(JITTER_BCS, color="k", ls="--", lw=1.2,
                label=f"value used in study ({JITTER_BCS})")
axes[0].set_xlabel("Colour-jitter strength (brightness/contrast/saturation)")
axes[0].set_ylabel("Test accuracy")
axes[0].set_title("Degradation curves across perturbation intensity")
axes[0].legend(fontsize=7.5, ncol=2, loc="lower left"); axes[0].grid(alpha=0.3)

im = axes[1].imshow(ranks.values, cmap="RdYlGn_r", aspect="auto")
axes[1].set_xticks(range(len(JITTER_LEVELS)))
axes[1].set_xticklabels([f"{s:.2f}" for s in JITTER_LEVELS])
axes[1].set_yticks(range(len(ranks))); axes[1].set_yticklabels(ranks.index, fontsize=8.5)
for i in range(ranks.shape[0]):
    for j in range(ranks.shape[1]):
        axes[1].text(j, i, int(ranks.values[i, j]), ha="center", va="center", fontsize=8)
axes[1].set_xlabel("Jitter strength"); axes[1].set_title("Model rank at each intensity")
plt.colorbar(im, ax=axes[1], fraction=0.035, label="rank (1 = best)")

plt.tight_layout()
plt.savefig(JIT_OUT / "jitter_sweep.png", dpi=300, bbox_inches="tight")
plt.show()

---
# 25. Proposed models under matched training regimes

R2-5 asks for the **proposed networks** under frozen / partially unfrozen /
fully fine-tuned settings — Section 19 covered only ResNet-50 and ConvNeXt-Tiny.

The regime classes below drop the `torch.no_grad()` wrapper so gradients can
reach the backbone when a regime requires it. With everything frozen the
mathematics is unchanged, so the `frozen` regime reproduces the original model.

Full fine-tuning of DPCT-Net trains two backbones at once; batch size is
reduced automatically if memory runs short.

In [ ]:
PREG_OUT = OUT_DIR / "proposed_regimes"
(PREG_OUT / "models").mkdir(parents=True, exist_ok=True)
(PREG_OUT / "metrics").mkdir(parents=True, exist_ok=True)


class DPCT_Regime(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, regime="frozen"):
        super().__init__()
        self.cnn = timm.create_model('convnext_tiny', pretrained=True,
                                     num_classes=0, global_pool='avg')
        self.tfm = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True,
                                     num_classes=0, global_pool='avg')
        for p in self.cnn.parameters(): p.requires_grad = False
        for p in self.tfm.parameters(): p.requires_grad = False
        if regime == "partial":
            if hasattr(self.cnn, "stages"):
                for p in self.cnn.stages[-1].parameters(): p.requires_grad = True
            if hasattr(self.tfm, "layers"):
                for p in self.tfm.layers[-1].parameters(): p.requires_grad = True
        elif regime == "full":
            for p in self.cnn.parameters(): p.requires_grad = True
            for p in self.tfm.parameters(): p.requires_grad = True
        d = 256
        self.proj_cnn = nn.Sequential(nn.Linear(self.cnn.num_features, d), nn.GELU(), nn.LayerNorm(d))
        self.proj_tfm = nn.Sequential(nn.Linear(self.tfm.num_features, d), nn.GELU(), nn.LayerNorm(d))
        self.cross_attn = nn.MultiheadAttention(d, num_heads=4, batch_first=True)
        self.gate = nn.Sequential(nn.Linear(d*2, d), nn.GELU(), nn.Linear(d, 2), nn.Softmax(dim=-1))
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(d, num_classes))

    def forward(self, x):
        f_cnn = self.cnn(x); f_tfm = self.tfm(x)
        z_c = self.proj_cnn(f_cnn); z_t = self.proj_tfm(f_tfm)
        seq = torch.stack([z_c, z_t], dim=1)
        ao, _ = self.cross_attn(seq, seq, seq)
        g = self.gate(torch.cat([z_c, z_t], dim=-1))
        return self.classifier(g[:, 0:1]*ao[:, 0] + g[:, 1:2]*ao[:, 1])


class MSCA_Regime(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, regime="frozen"):
        super().__init__()
        self.backbone = timm.create_model('convnext_tiny', pretrained=True,
                                          features_only=True, out_indices=(1, 2, 3))
        for p in self.backbone.parameters(): p.requires_grad = False
        if regime == "partial":
            mods = [m for m in self.backbone.modules() if isinstance(m, nn.Conv2d)]
            for p in mods[-8:]:
                for q in p.parameters(): q.requires_grad = True
        elif regime == "full":
            for p in self.backbone.parameters(): p.requires_grad = True
        chs = self.backbone.feature_info.channels(); d = 256
        self.proj = nn.ModuleList([nn.Sequential(nn.Conv2d(c, d, 1), nn.BatchNorm2d(d), nn.GELU())
                                   for c in chs])
        self.ca = nn.ModuleList([nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(d, d//8, 1), nn.ReLU(inplace=True),
            nn.Conv2d(d//8, d, 1), nn.Sigmoid()) for _ in chs])
        self.spatial_attn = nn.ModuleList([nn.Sequential(nn.Conv2d(d, 1, 1), nn.Sigmoid())
                                           for _ in chs])
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.scale_fusion = nn.Sequential(nn.Linear(d*len(chs), d), nn.GELU(), nn.LayerNorm(d))
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(d, num_classes))

    def forward(self, x):
        feats_ = self.backbone(x)
        proj = [p(f) for p, f in zip(self.proj, feats_)]
        ref = [None]*len(proj)
        ref[-1] = proj[-1] * self.ca[-1](proj[-1])
        for i in range(len(proj)-2, -1, -1):
            a = self.spatial_attn[i+1](ref[i+1])
            a = F.interpolate(a, size=proj[i].shape[2:], mode='bilinear', align_corners=False)
            ref[i] = proj[i] * self.ca[i](proj[i]) * a
        pooled = torch.cat([self.gap(r).flatten(1) for r in ref], dim=1)
        return self.classifier(self.scale_fusion(pooled))


print("Regime-capable variants defined")
for cls, nm in [(DPCT_Regime, "DPCT"), (MSCA_Regime, "MSCA")]:
    for rg in ["frozen", "partial", "full"]:
        m = cls(regime=rg); t, tr = count_parameters(m)
        print(f"  {nm}_{rg:8s} total {t/1e6:6.2f}M  trainable {tr/1e6:7.3f}M ({100*tr/t:5.2f}%)")
        del m

In [ ]:
def loaders_at_batch(bs):
    return (DataLoader(SplitDataset(lc_df, "train", CLASS_NAMES, train_transform),
                       batch_size=bs, shuffle=True, num_workers=0, pin_memory=False),
            DataLoader(SplitDataset(lc_df, "val", CLASS_NAMES, eval_transform),
                       batch_size=bs, shuffle=False, num_workers=0, pin_memory=False))


PREG_CSV = PREG_OUT / "proposed_regimes.csv"
preg_rows = pd.read_csv(PREG_CSV).to_dict("records") if PREG_CSV.exists() else []
done_p = {(r["Architecture"], r["Regime"]) for r in preg_rows}
print(f"Resuming — {len(done_p)} runs recorded")

for cls, arch in [(DPCT_Regime, "DPCT-Net"), (MSCA_Regime, "MSCA-Net")]:
    for regime in ["frozen", "partial", "full"]:
        if (arch, regime) in done_p:
            print(f"  {arch} {regime}: already done"); continue
        tag = f"{arch.replace('-','')}_{regime}"
        print(f"\n{'='*70}\n  {arch} :: {regime}\n{'='*70}")

        bs = BATCH_SIZE
        for attempt in range(3):
            try:
                torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                model = cls(num_classes=NUM_CLASSES, regime=regime)
                tot, tr = count_parameters(model)
                tl, vl = loaders_at_batch(bs)
                model, _ = train_model(model, tag, tl, vl, PREG_OUT, "PREG")
                m = evaluate_model(model, lr_test_loader, tag)
                pert = evaluate_model(model, lr_stain_loader, tag)["accuracy"]
                preg_rows.append({"Architecture": arch, "Regime": regime,
                                  "batch": bs,
                                  "Total (M)": round(tot/1e6, 2),
                                  "Trainable (M)": round(tr/1e6, 3),
                                  "Trainable %": round(100*tr/tot, 2),
                                  "Accuracy": round(m["accuracy"], 4),
                                  "F1": round(m["f1_macro"], 4),
                                  "Perturbed": round(pert, 4)})
                pd.DataFrame(preg_rows).to_csv(PREG_CSV, index=False)
                print(f"  acc={m['accuracy']:.4f}  perturbed={pert:.4f}  "
                      f"trainable={tr/1e6:.3f}M  [saved]")
                del model, tl, vl
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                break
            except torch.cuda.OutOfMemoryError:
                bs = max(4, bs // 2)
                print(f"  OOM — retrying at batch {bs}")
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

preg_df = pd.DataFrame(preg_rows)
print()
print(preg_df.to_string(index=False))

In [ ]:
# combine with Section 19 so all four architectures appear together
if "reg_df" in dir() and len(reg_df):
    base = reg_df.rename(columns={"Architecture": "Architecture"}).copy()
    base["batch"] = BATCH_SIZE
    allreg = pd.concat([base, preg_df], ignore_index=True, sort=False)
else:
    allreg = preg_df.copy()
allreg.to_csv(OUT_DIR / "all_matched_regimes.csv", index=False)
print(allreg.to_string(index=False))

order = ["linear_probe", "frozen", "partial", "full"]
present = [r for r in order if r in set(allreg["Regime"])]
fig, ax = plt.subplots(figsize=(12, 5.8))
archs = list(dict.fromkeys(allreg["Architecture"]))
x = np.arange(len(present)); w = 0.8/len(archs)
for k, a in enumerate(archs):
    sub = allreg[allreg["Architecture"] == a].set_index("Regime")
    vals = [sub.loc[r, "Accuracy"] if r in sub.index else np.nan for r in present]
    trs  = [sub.loc[r, "Trainable (M)"] if r in sub.index else np.nan for r in present]
    ax.bar(x + (k - (len(archs)-1)/2)*w, vals, w, label=a, alpha=0.9)
    for xi, (v, t) in enumerate(zip(vals, trs)):
        if not np.isnan(v):
            ax.text(xi + (k - (len(archs)-1)/2)*w, v + 0.003,
                    f"{v:.4f}\n{t:.2f}M", ha="center", fontsize=7)
ax.set_xticks(x); ax.set_xticklabels([p.replace("_", " ") for p in present])
ax.set_ylabel("Test accuracy (leakage-resistant)")
ax.set_title("Matched training regimes — all four architectures")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
lo = np.nanmin(allreg["Accuracy"]); ax.set_ylim(max(0, lo-0.06), 1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "all_matched_regimes.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nRegime sensitivity per architecture:")
for a in archs:
    s = allreg[allreg["Architecture"] == a]
    print(f"  {a:16s} {s['Accuracy'].min():.4f} -> {s['Accuracy'].max():.4f} "
          f"({(s['Accuracy'].max()-s['Accuracy'].min())*100:5.2f} pp)")

In [ ]:
# 1. Merge the two regimes to fix the giant gap
allreg["Regime"] = allreg["Regime"].replace({
    "linear_probe": "Linear probe / Frozen", 
    "frozen": "Linear probe / Frozen"
})

order = ["Linear probe / Frozen", "partial", "full"]
present = [r for r in order if r in set(allreg["Regime"])]

fig, ax = plt.subplots(figsize=(10, 6.5)) # Slightly taller for text breathing room
archs = list(dict.fromkeys(allreg["Architecture"]))
x = np.arange(len(present)); w = 0.8/len(archs)

# 2. Apply a modern, cohesive color palette
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

for k, a in enumerate(archs):
    sub = allreg[allreg["Architecture"] == a].set_index("Regime")
    vals = [sub.loc[r, "Accuracy"] if r in sub.index else np.nan for r in present]
    trs = [sub.loc[r, "Trainable (M)"] if r in sub.index else np.nan for r in present]
    
    # Added edgecolors to make the bars pop and look distinct
    ax.bar(x + (k - (len(archs)-1)/2)*w, vals, w, label=a, 
           color=colors[k % len(colors)], edgecolor="white", linewidth=1.2, zorder=3)
    
    for xi, (v, t) in enumerate(zip(vals, trs)):
        if not np.isnan(v):
            # Adjusted vertical alignment (va) to sit perfectly above the bar
            ax.text(xi + (k - (len(archs)-1)/2)*w, v + 0.002, 
                    f"{v:.4f}\n{t:.2f}M", ha="center", va="bottom", fontsize=8)

# 3. Clean up the axes and labels
ax.set_xticks(x)
ax.set_xticklabels([p.replace("_", " ").title() for p in present], fontsize=10, fontweight="bold")
ax.set_ylabel("Test accuracy (leakage-resistant)", fontsize=11)
ax.set_title("Matched training regimes — all four architectures", fontsize=13, pad=15)

# 4. Remove ugly borders and format the grid
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#DDDDDD')
ax.spines['bottom'].set_color('#DDDDDD')

# Frame-less legend for a cleaner look
ax.legend(fontsize=9, loc="upper left", frameon=False) 

# Dashed grid lines pushed behind the bars (using zorder=0)
ax.grid(axis="y", alpha=0.4, linestyle="--", zorder=0)
ax.grid(axis="x", visible=False) # Vertical grids are distracting here

lo = np.nanmin(allreg["Accuracy"])
ax.set_ylim(max(0, lo-0.03), 1.01) # Tighter Y-axis padding

plt.tight_layout()
# plt.savefig(OUT_DIR / "figures" / "all_matched_regimes1.png", dpi=600, bbox_inches="tight")
plt.show()

print("\nRegime sensitivity per architecture:")
for a in archs:
    s = allreg[allreg["Architecture"] == a]
    print(f"  {a:16s} {s['Accuracy'].min():.4f} -> {s['Accuracy'].max():.4f} "
          f"({(s['Accuracy'].max()-s['Accuracy'].min())*100:5.2f} pp)")

---
# 26. Part III summary

In [ ]:
print("="*72); print("PART III SUMMARY"); print("="*72)

if "cost_df" in dir():
    print("\n[23] COMPUTATIONAL COST")
    c = cost_df.dropna(subset=["GFLOPs"])
    if len(c):
        print(f"  Lowest GFLOPs : {c.loc[c['GFLOPs'].idxmin(),'Model']} "
              f"({c['GFLOPs'].min():.2f})")
        print(f"  Highest GFLOPs: {c.loc[c['GFLOPs'].idxmax(),'Model']} "
              f"({c['GFLOPs'].max():.2f})")
    for n in ["DPCT_Net", "MSCA_Net"]:
        if n in cost_df["Model"].values:
            r = cost_df[cost_df["Model"] == n].iloc[0]
            print(f"  {n}: {r.get('GFLOPs', float('nan'))} GFLOPs, "
                  f"{r.get('Trainable (M)')}M trainable, "
                  f"{r.get('peak VRAM MB @bs32', float('nan'))} MB peak VRAM")

if "rho_df" in dir():
    print("\n[24] JITTER CALIBRATION")
    print(f"  Mean Spearman rho across intensities: {rho_df['spearman rho'].mean():.4f}")
    print("  -> ranking is " + ("stable" if rho_df['spearman rho'].mean() > 0.8
                                 else "NOT stable") + " across perturbation strengths")

if "allreg" in dir():
    print("\n[25] MATCHED REGIMES (all architectures)")
    for a in dict.fromkeys(allreg["Architecture"]):
        s = allreg[allreg["Architecture"] == a]
        best = s.loc[s["Accuracy"].idxmax()]
        print(f"  {a:16s} best = {best['Regime']:12s} {best['Accuracy']:.4f}")
print("="*72)